In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#read in dataset. 
with open('imputed_study_data.pkl', 'rb') as f:
    df=pickle.load(f)

In [3]:
#convert to daily values 
df['date'] = df['time'].dt.tz_convert(None).dt.date
agg_cols = [
    'value',
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'wind_speed_10m','wind_direction_10m','surface_pressure','precipitation'
]
grouped = df.groupby(['location_id','station_lat','station_lon','date'], as_index=False)[agg_cols].mean()

grouped.head(5)

,location_id,station_lat,station_lon,date,value,temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,wind_direction_10m,surface_pressure,precipitation
0,2622586,37.580167,127.044856,2024-12-01,20.541667,0.770833,95.375000,0.062500,2.983333,94.583333,999.987500,0.012500
1,2622586,37.580167,127.044856,2024-12-02,21.416667,4.108333,83.958333,1.487500,7.937500,206.416667,999.816667,0.020833
2,2622586,37.580167,127.044856,2024-12-03,6.583333,-0.920833,59.791667,-8.079167,6.525000,269.875000,1007.179167,0.000000
3,2622586,37.580167,127.044856,2024-12-04,11.128392,0.145833,66.041667,-5.887500,4.466667,290.041667,1006.158333,0.000000
4,2622586,37.580167,127.044856,2024-12-05,8.666667,1.191667,72.083333,-3.695833,5.216667,256.791667,1001.008333,0.000000


In [4]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculates the great-circle distance between two points in km."""
    R = 6371.0  # Earth radius in kilometers
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def bearing_deg(lat1, lon1, lat2, lon2):
    """Calculates the initial bearing from point 1 to point 2."""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    y = math.sin(lon2 - lon1) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - \
        math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
    
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def angle_diff_deg(angle1, angle2):
    """Calculates the smallest angular difference between two angles (0-360)."""
    diff = abs(angle1 - angle2) % 360
    return min(diff, 360 - diff)

In [6]:
# 1. Define predictors
import networkx as nx
predictor_cols = [
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'wind_speed_10m','wind_direction_10m','surface_pressure','precipitation'
]

# 2. Build directed graphs
graphs = {}
DIST_THRESHOLD_KM = 5.0

for d in grouped['date'].unique():
    date_key = d.isoformat() if hasattr(d, 'isoformat') else str(d)
    day = grouped[grouped['date'] == d].reset_index(drop=True)
    G = nx.DiGraph(date=date_key)

    # --- REGIONAL BASELINE (Anomaly Calculation) ---
    raw_day_feats = day[predictor_cols].values
    regional_baseline = np.nanmean(raw_day_feats, axis=0)
    
    # --- NODE CONSTRUCTION ---
    for _, row in day.iterrows():
        loc = row['location_id']
        
        # Calculate raw features and impute NaNs with regional baseline to avoid errors
        raw_feat = np.array([float(row[col]) if col in row and not pd.isna(row[col]) else np.nan for col in predictor_cols])
        filled_feat = np.where(np.isnan(raw_feat), regional_baseline, raw_feat)
        
        # Anomaly = Deviation from spatial mean
        anomaly_feat = filled_feat - regional_baseline
        
        # FINAL FEATURES: Lat/Lon are NOT appended here. 
        # The model only sees the weather anomalies.
        final_feature_vector = anomaly_feat

        G.add_node(loc,
                   station_lat=float(row['station_lat']),
                   station_lon=float(row['station_lon']),
                   features=final_feature_vector, 
                   feature_names=predictor_cols)

    # --- EDGE CONSTRUCTION (Lat/Lon used here for Topology) ---
    # We still use lat/lon here to ensure the 5km constraint
    n = len(day)
    for i in range(n):
        ri = day.loc[i]
        src = ri['location_id']
        wind_speed_i = float(ri['wind_speed_10m']) if not pd.isna(ri['wind_speed_10m']) else 0.0
        for j in range(n):
            if i == j: continue
            rj = day.loc[j]
            
            dist = haversine_km(float(ri['station_lat']), float(ri['station_lon']),
                                float(rj['station_lat']), float(rj['station_lon']))
            
            if dist <= DIST_THRESHOLD_KM:
                bearing = bearing_deg(float(ri['station_lat']), float(ri['station_lon']),
                                       float(rj['station_lat']), float(rj['station_lon']))
                diff = angle_diff_deg(float(ri['wind_direction_10m']), bearing)
                
                # Wind-driven edge weight
                score = max(math.cos(math.radians(diff)), 0.0) * wind_speed_i
                
                if score > 0:
                    G.add_edge(src, rj['location_id'], weight=score, distance_km=dist)

        # If wind is zero, preserve adjacency through a self-loop identity edge
        if wind_speed_i == 0.0 and not G.has_edge(src, src):
            G.add_edge(src, src, weight=1.0, distance_km=0.0)

    graphs[date_key] = G

In [7]:
import numpy as np
dates = sorted(list(graphs.keys()))
print("--- GRAPH DYNAMICS VERIFICATION ---")

# Track values for consecutive day comparisons
consecutive_dates = dates[:10]  # Look at the first 10 days as a sample

for i in range(len(consecutive_dates) - 1):
    d1, d2 = consecutive_dates[i], consecutive_dates[i+1]
    g1, g2 = graphs[d1], graphs[d2]
    
    # 1. Edge Sets
    edges1 = set(g1.edges())
    edges2 = set(g2.edges())
    
    # 2. Track weight variance for persistent edges
    shared_edges = edges1.intersection(edges2)
    weight_changes = []
    
    for u, v in shared_edges:
        w1 = g1[u][v]['weight']
        w2 = g2[u][v]['weight']
        weight_changes.append(abs(w1 - w2))
    
    # Calculate Jaccard Overlap for topology
    all_edges = edges1.union(edges2)
    jaccard = len(shared_edges) / len(all_edges) if all_edges else 1.0
    
    # Calculate mean weight shift for persistent paths
    mean_weight_shift = np.mean(weight_changes) if weight_changes else 0.0
    
    print(f"Shift from {d1} ➔ {d2}:")
    print(f"  • Total Edges      : Day1 = {g1.number_of_edges():<4} | Day2 = {g2.number_of_edges():<4}")
    print(f"  • Edge Jaccard     : {jaccard:.3f} (An overlap of 1.0 means static topology, closer to 0 means highly dynamic)")
    print(f"  • Mean Weight Shift: {mean_weight_shift:.4f} (Fluctuation in wind flux intensity for surviving edges)")
    print("-" * 50)

--- GRAPH DYNAMICS VERIFICATION ---
Shift from 2024-12-01 ➔ 2024-12-02:
  • Total Edges      : Day1 = 54   | Day2 = 55  
  • Edge Jaccard     : 0.198 (An overlap of 1.0 means static topology, closer to 0 means highly dynamic)
  • Mean Weight Shift: 4.3594 (Fluctuation in wind flux intensity for surviving edges)
--------------------------------------------------
Shift from 2024-12-02 ➔ 2024-12-03:
  • Total Edges      : Day1 = 55   | Day2 = 54  
  • Edge Jaccard     : 0.603 (An overlap of 1.0 means static topology, closer to 0 means highly dynamic)
  • Mean Weight Shift: 3.8107 (Fluctuation in wind flux intensity for surviving edges)
--------------------------------------------------
Shift from 2024-12-03 ➔ 2024-12-04:
  • Total Edges      : Day1 = 54   | Day2 = 54  
  • Edge Jaccard     : 0.800 (An overlap of 1.0 means static topology, closer to 0 means highly dynamic)
  • Mean Weight Shift: 2.1677 (Fluctuation in wind flux intensity for surviving edges)
-------------------------------

In [ ]:
#now, we standardize our observed nodes features by making a daily baseline feature vector, then
#we use this along with the observed nodes features to make the unit equal. 

#specifically, if the baseline for temperature on day 1 is 20, and my nodes value is 22, the temperature
#for node becomes 2 etc. 

In [9]:
#now extract a grid around the original region to sample values. 



import os
import time
import math
import random
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta, datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
from threading import Lock

# -------------------------
# User inputs (edit as needed)
# -------------------------
BBOX = (126.8, 37.4, 127.1, 37.65)   # (min_lon, min_lat, max_lon, max_lat)
lengthscale_km = 4.5                 # spatial lengthscale in km (used to set spacing if building grid)
spacing_km = lengthscale_km / 2.0
cell_size_m = spacing_km * 1000.0

# Try to use existing grid_centers or centers; otherwise build grid from BBOX
try:
    grid_centers  # noqa: F821
    use_existing_grid = True
except NameError:
    try:
        centers  # noqa: F821
        grid_centers = centers
        use_existing_grid = True
    except NameError:
        use_existing_grid = False

# -------------------------
# Open‑Meteo daily endpoint and variables (deduped)
# -------------------------
ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
daily_vars = [
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "dew_point_2m_mean",
    "windspeed_10m_mean",
    "winddirection_10m_dominant",
    "surface_pressure_mean",
    "precipitation_sum"
]
# ensure uniqueness while preserving order
_seen = set()
daily_vars = [x for x in daily_vars if not (x in _seen or _seen.add(x))]

timezone = "UTC"

# Date range (uses your provided last timestamp)
first_timestamp = pd.Timestamp('2024-12-01 00:00:00+0000', tz='UTC')
start_date = first_timestamp.tz_convert("UTC").date().isoformat()
last_timestamp = pd.Timestamp("2025-08-31 23:00:00+0000", tz="UTC")
end_date = last_timestamp.tz_convert("UTC").date().isoformat()

# Output and fetch settings (safer defaults)
out_dir = "open_meteo_grid_daily_direct"
os.makedirs(out_dir, exist_ok=True)
requests_per_second = 1.0   # safer default; increase only if you confirm API capacity
max_workers = 4            # lower concurrency to reduce bursts
max_retries = 5
retry_backoff_base = 2.0   # base for exponential backoff
max_backoff_seconds = 300  # cap backoff to 5 minutes

# -------------------------
# Build grid_centers if needed (square metric grid using AEQD projection)
# -------------------------
if not use_existing_grid:
    min_lon, min_lat, max_lon, max_lat = BBOX
    center_lon = (min_lon + max_lon) / 2.0
    center_lat = (min_lat + max_lat) / 2.0

    proj_str = f"+proj=aeqd +lat_0={center_lat} +lon_0={center_lon} +units=m +datum=WGS84 +no_defs"
    transformer_to_m = Transformer.from_crs("epsg:4326", proj_str, always_xy=True)
    transformer_to_lonlat = Transformer.from_crs(proj_str, "epsg:4326", always_xy=True)

    def lonlat_to_m(lon, lat):
        x, y = transformer_to_m.transform(lon, lat)
        return float(x), float(y)

    def m_to_lonlat(x, y):
        lon, lat = transformer_to_lonlat.transform(x, y)
        return float(lon), float(lat)

    x_min, y_min = lonlat_to_m(min_lon, min_lat)
    x_max, y_max = lonlat_to_m(max_lon, max_lat)
    x0, x1 = min(x_min, x_max), max(x_min, x_max)
    y0, y1 = min(y_min, y_max), max(y_min, y_max)

    n_cols = int(math.ceil((x1 - x0) / cell_size_m))
    n_rows = int(math.ceil((y1 - y0) / cell_size_m))
    grid_centers = []
    for i in range(n_cols):
        for j in range(n_rows):
            x_left = x0 + i * cell_size_m
            x_right = x0 + (i + 1) * cell_size_m
            y_bottom = y0 + j * cell_size_m
            y_top = y0 + (j + 1) * cell_size_m
            cx = (x_left + x_right) / 2.0
            cy = (y_bottom + y_top) / 2.0
            lonc, latc = m_to_lonlat(cx, cy)
            grid_centers.append((lonc, latc))
    print(f"Built grid: {n_cols} cols × {n_rows} rows = {len(grid_centers)} cells; spacing {spacing_km:.3f} km")
else:
    # normalize grid_centers formats
    if isinstance(grid_centers, np.ndarray):
        grid_centers = [tuple(x) for x in grid_centers.tolist()]
    elif isinstance(grid_centers, pd.DataFrame):
        if {'lon','lat'}.issubset(set(grid_centers.columns)):
            grid_centers = list(zip(grid_centers['lon'].values, grid_centers['lat'].values))
        else:
            raise RuntimeError("If grid_centers is a DataFrame it must contain 'lon' and 'lat' columns.")
    print(f"Using existing grid_centers with {len(grid_centers)} cells")

# Quick validation: ensure tuples are (lon, lat). If many entries look swapped, offer automatic swap.
def looks_like_latlon_swapped(sample):
    lon, lat = sample
    return abs(lon) <= 90 and abs(lat) <= 180 and not (-180 <= lon <= 180 and -90 <= lat <= 90)

if len(grid_centers) > 0:
    first = grid_centers[0]
    if looks_like_latlon_swapped(first):
        print("Detected grid_centers likely in (lat, lon) order; swapping to (lon, lat).")
        grid_centers = [(lonlat[1], lonlat[0]) for lonlat in grid_centers]

# -------------------------
# Helpers for daily fetch with retries, 429 handling, and caching
# -------------------------
def build_params_daily(lat, lon, start_iso, end_iso, daily_vars, timezone="UTC"):
    return {
        "latitude": float(lat),
        "longitude": float(lon),
        "start_date": start_iso,
        "end_date": end_iso,
        "daily": ",".join(daily_vars),
        "timezone": timezone
    }

SESSION = requests.Session()
RATE_LOCK = Lock()
LAST_REQUEST_TS = 0.0

def throttle():
    """Global pacing to respect requests_per_second across threads."""
    global LAST_REQUEST_TS
    with RATE_LOCK:
        min_interval = 1.0 / max(1.0, requests_per_second)
        now = time.time()
        wait = LAST_REQUEST_TS + min_interval - now
        if wait > 0:
            time.sleep(wait)
        LAST_REQUEST_TS = time.time()

def _sleep_with_jitter(seconds):
    """Sleep with small jitter to avoid synchronized retries."""
    jitter = random.uniform(0.0, 0.25 * seconds) if seconds > 0 else 0.0
    time.sleep(seconds + jitter)

def fetch_daily_with_retries(lat, lon, start_iso, end_iso, daily_vars, timezone="UTC"):
    params = build_params_daily(lat, lon, start_iso, end_iso, daily_vars, timezone)
    attempt = 0
    while attempt <= max_retries:
        try:
            throttle()
            r = SESSION.get(ARCHIVE_BASE, params=params, timeout=60)
            if r.status_code == 200:
                try:
                    return r.json()
                except ValueError:
                    raise RuntimeError(f"Invalid JSON response for ({lat},{lon})")
            elif r.status_code == 429:
                # Rate limited: honor Retry-After if present, otherwise exponential backoff
                retry_after = r.headers.get("Retry-After")
                if retry_after is not None:
                    try:
                        wait = float(retry_after)
                    except Exception:
                        # sometimes Retry-After is a HTTP-date; fallback to a safe wait
                        wait = min(60.0, retry_backoff_base ** (attempt + 1))
                    wait = min(wait, max_backoff_seconds)
                    print(f"429 for ({lat},{lon}) — server asked to wait {wait}s (Retry-After). Attempt {attempt+1}/{max_retries}")
                    _sleep_with_jitter(wait)
                else:
                    # exponential backoff with jitter
                    wait = min(max_backoff_seconds, retry_backoff_base ** (attempt + 1))
                    print(f"429 for ({lat},{lon}) — backing off {wait}s (no Retry-After). Attempt {attempt+1}/{max_retries}")
                    _sleep_with_jitter(wait)
                attempt += 1
                continue
            else:
                text = r.text[:1000] if r.text else ""
                print(f"API returned status {r.status_code} for ({lat},{lon}) start={start_iso} end={end_iso}: {text}")
                r.raise_for_status()
        except requests.RequestException as e:
            attempt += 1
            # exponential backoff with jitter for network errors
            wait = min(max_backoff_seconds, retry_backoff_base ** attempt)
            print(f"Network/request error for ({lat},{lon}) attempt {attempt}/{max_retries}: {e}. Backing off {wait}s")
            _sleep_with_jitter(wait)
            if attempt > max_retries:
                raise RuntimeError(f"Failed to fetch daily ({lat},{lon}) after {max_retries} retries: {e}")
        except Exception as e:
            attempt += 1
            wait = min(max_backoff_seconds, retry_backoff_base ** attempt)
            print(f"Unexpected error for ({lat},{lon}) attempt {attempt}/{max_retries}: {e}. Backing off {wait}s")
            _sleep_with_jitter(wait)
            if attempt > max_retries:
                raise RuntimeError(f"Failed to fetch daily ({lat},{lon}) after {max_retries} retries: {e}")
    raise RuntimeError("unreachable")

def safe_daily_path(cell_dir, idx):
    return os.path.join(cell_dir, f"cell_{idx:04d}_daily")

# -------------------------
# Parse daily payload into DataFrame (handles missing vars gracefully)
# -------------------------
def parse_daily_payload(payload, daily_vars):
    daily = payload.get("daily", {})
    times = daily.get("time", [])
    if len(times) == 0:
        return pd.DataFrame(columns=["date"] + daily_vars)
    try:
        df = pd.DataFrame({"date": pd.to_datetime(times)})
        for var in daily_vars:
            vals = daily.get(var)
            if vals is None:
                df[var] = np.nan
            else:
                df[var] = pd.to_numeric(vals, errors="coerce")
        df['date'] = pd.to_datetime(df['date']).dt.date
        return df
    except Exception as e:
        print("Failed to parse payload into DataFrame:", e)
        try:
            print("Payload daily keys:", list(daily.keys()))
            import json
            sample = {k: (daily.get(k)[:3] if isinstance(daily.get(k), list) else daily.get(k)) for k in list(daily.keys())[:10]}
            print(json.dumps(sample, default=str))
        except Exception:
            pass
        raise

# -------------------------
# Main: fetch full-range daily per cell (one request per cell), cache per-cell daily parquet/csv
# -------------------------
n_cells = len(grid_centers)
print(f"Fetching Open‑Meteo daily for {n_cells} cells from {start_date} to {end_date} (one request per cell)")

# pre-create date keys
start_dt = pd.to_datetime(start_date).date()
end_dt = pd.to_datetime(end_date).date()
all_dates = pd.date_range(start=start_dt, end=end_dt, freq="D").date.tolist()
data_by_day = {d.isoformat(): {} for d in all_dates}

def process_cell_daily(idx, lon, lat):
    cell_dir = os.path.join(out_dir, f"cell_{idx:04d}")
    os.makedirs(cell_dir, exist_ok=True)
    base_path = safe_daily_path(cell_dir, idx)
    parquet_path = base_path + ".parquet"
    csv_path = base_path + ".csv"

    # if cached parquet or csv exists, load and return
    if os.path.exists(parquet_path):
        try:
            df_cell = pd.read_parquet(parquet_path)
            if 'date' in df_cell.columns:
                df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date
            return idx, lon, lat, df_cell
        except Exception as e:
            print(f"Warning: failed to read parquet for cell {idx}: {e}")

    if os.path.exists(csv_path):
        try:
            df_cell = pd.read_csv(csv_path, parse_dates=["date"])
            if 'date' in df_cell.columns:
                df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date
            return idx, lon, lat, df_cell
        except Exception as e:
            print(f"Warning: failed to read csv for cell {idx}: {e}")

    # sanity check for coordinate ranges
    if not (-90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0):
        raise RuntimeError(f"Invalid coordinates for cell {idx}: lat={lat}, lon={lon}")

    # fetch daily range in one request (with robust retry/429 handling)
    payload = fetch_daily_with_retries(lat=lat, lon=lon, start_iso=start_date, end_iso=end_date, daily_vars=daily_vars, timezone=timezone)

    df_cell = parse_daily_payload(payload, daily_vars)

    # if API returned no rows, create full-range empty frame
    if df_cell.shape[0] == 0:
        df_cell = pd.DataFrame({"date": all_dates})
        for var in daily_vars:
            df_cell[var] = np.nan
    else:
        df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date

    # -------------------------
    # Safe reindexing: avoid duplicate 'date' column
    # -------------------------
    if 'date' in df_cell.columns:
        temp_index = pd.to_datetime(df_cell['date'])
        df_cell = df_cell.drop(columns=['date'])
    else:
        temp_index = pd.to_datetime(df_cell.index)

    df_cell = df_cell.set_index(temp_index).reindex(pd.to_datetime(all_dates))
    df_cell.index.name = 'date'
    df_cell = df_cell.reset_index()
    df_cell['date'] = pd.to_datetime(df_cell['date']).dt.date

    # add lon/lat columns
    df_cell['lon'] = lon
    df_cell['lat'] = lat

    # try to save parquet, fallback to CSV
    try:
        df_cell.to_parquet(parquet_path, index=False)
    except Exception as e:
        try:
            df_cell.to_csv(csv_path, index=False)
        except Exception as e2:
            print(f"Failed to save cell {idx} to parquet and csv: {e2}")

    return idx, lon, lat, df_cell

# run cells in parallel (one request per cell)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(process_cell_daily, idx, lon, lat): (idx, lon, lat) for idx, (lon, lat) in enumerate(grid_centers)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching cells"):
        idx, lon, lat = futures[fut]
        try:
            res_idx, res_lon, res_lat, df_cell = fut.result()
        except Exception as e:
            print(f"Cell {idx} failed: {e}")
            continue
        if df_cell is None or df_cell.shape[0] == 0:
            continue
        # populate data_by_day from df_cell
        for _, row in df_cell.iterrows():
            date_iso = pd.to_datetime(row['date']).date().isoformat()
            if date_iso not in data_by_day:
                continue
            entry = {'lon': float(res_lon), 'lat': float(res_lat)}
            for var in daily_vars:
                entry[var] = float(row[var]) if var in row and not pd.isna(row[var]) else np.nan
            data_by_day[date_iso][int(res_idx)] = entry

print("Done. Cached per-cell daily files are in:", out_dir)
print("Access daily predictors via data_by_day[date_iso][cell_idx] -> dict of daily features")



Built grid: 12 cols × 13 rows = 156 cells; spacing 2.250 km
Fetching Open‑Meteo daily for 156 cells from 2024-12-01 to 2025-08-31 (one request per cell)


Fetching cells: 100%|██████████| 156/156 [00:01<00:00, 86.33it/s] 

Done. Cached per-cell daily files are in: open_meteo_grid_daily_direct
Access daily predictors via data_by_day[date_iso][cell_idx] -> dict of daily features


In [10]:
#standardize the grid values 

# Map Open-Meteo daily names to your network predictor names
grid_var_map = {
    "temperature_2m_mean": "temperature_2m",
    "relative_humidity_2m_mean": "relative_humidity_2m",
    "dew_point_2m_mean": "dew_point_2m",
    "windspeed_10m_mean": "wind_speed_10m",
    "winddirection_10m_dominant": "wind_direction_10m",
    "surface_pressure_mean": "surface_pressure",
    "precipitation_sum": "precipitation",
}

predictor_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "precipitation",
]

# Build standardized grid features by day
grid_features_by_day = {}

for date_iso, cells in data_by_day.items():
    if not cells:
        continue

    # Build a DataFrame of grid cells for this day
    df_grid = pd.DataFrame.from_dict(cells, orient="index")
    df_grid = df_grid.rename(columns=grid_var_map)

    # Ensure all predictor cols exist
    for col in predictor_cols:
        if col not in df_grid.columns:
            df_grid[col] = np.nan

    raw_feats = df_grid[predictor_cols].to_numpy(dtype=float)

    # Regional baseline across the grid for this date
    regional_baseline = np.nanmean(raw_feats, axis=0)

    # Fill missing values with baseline, then compute anomalies
    filled_feats = np.where(np.isnan(raw_feats), regional_baseline, raw_feats)
    anomaly_feats = filled_feats - regional_baseline

    day_grid = {}
    for cell_idx, row in df_grid.iterrows():
        idx = int(cell_idx)
        lon = float(row["lon"])
        lat = float(row["lat"])

        feature_vector = anomaly_feats[list(df_grid.index).index(cell_idx)]
        day_grid[idx] = {
            "lon": lon,
            "lat": lat,
            "features": feature_vector,
            "feature_names": predictor_cols,
        }

    grid_features_by_day[date_iso] = day_grid

# Example access:
# grid_features_by_day["2024-12-01"][0]["features"]

In [11]:
grid_features_by_day['2024-12-01']

{2: {'lon': 126.81261298809714,
  'lat': 37.45069761656908,
  'features': array([ 0.50769231, -0.52564103,  0.42115385,  1.66217949, 12.03205128,
          6.38525641, -0.07884615]),
  'feature_names': ['temperature_2m',
   'relative_humidity_2m',
   'dew_point_2m',
   'wind_speed_10m',
   'wind_direction_10m',
   'surface_pressure',
   'precipitation']},
 3: {'lon': 126.81257587462069,
  'lat': 37.47097035524067,
  'features': array([ 0.80769231,  1.47435897,  1.02115385,  2.56217949, 11.03205128,
          2.68525641,  0.02115385]),
  'feature_names': ['temperature_2m',
   'relative_humidity_2m',
   'dew_point_2m',
   'wind_speed_10m',
   'wind_direction_10m',
   'surface_pressure',
   'precipitation']},
 1: {'lon': 126.81265007013226,
  'lat': 37.43042480815787,
  'features': array([ 0.50769231, -0.52564103,  0.42115385,  1.66217949, 12.03205128,
          6.88525641, -0.07884615]),
  'feature_names': ['temperature_2m',
   'relative_humidity_2m',
   'dew_point_2m',
   'wind_speed_10

In [12]:
# Compare each grid cell to its nearest existing station on the same date
import numpy as np
import pandas as pd

# Build a lookup for station nodes per date
station_nodes_by_date = {}
for date_iso, G in graphs.items():
    station_nodes_by_date[date_iso] = [
        {
            "location_id": node,
            "lat": attrs["station_lat"],
            "lon": attrs["station_lon"],
            "features": np.asarray(attrs["features"], dtype=float),
        }
        for node, attrs in G.nodes(data=True)
    ]

records = []
for date_iso, grid_cells in grid_features_by_day.items():
    if date_iso not in station_nodes_by_date:
        continue

    stations = station_nodes_by_date[date_iso]
    if len(stations) == 0:
        continue

    for cell_idx, cell_data in grid_cells.items():
        grid_lat = float(cell_data["lat"])
        grid_lon = float(cell_data["lon"])
        grid_vec = np.asarray(cell_data["features"], dtype=float)

        # find nearest station
        best = min(
            stations,
            key=lambda s: haversine_km(grid_lat, grid_lon, s["lat"], s["lon"])
        )
        dist_km = haversine_km(grid_lat, grid_lon, best["lat"], best["lon"])
        station_vec = best["features"]

        vec_diff = grid_vec - station_vec
        abs_diff = np.abs(vec_diff)

        rec = {
            "date": date_iso,
            "cell_idx": int(cell_idx),
            "nearest_station": best["location_id"],
            "station_distance_km": dist_km,
            "l2_diff": np.linalg.norm(vec_diff),
            "max_abs_diff": np.max(abs_diff),
            "mean_abs_diff": np.mean(abs_diff),
        }
        for name, diff in zip(predictor_cols, abs_diff):
            rec[f"{name}_abs_diff"] = diff

        records.append(rec)

df_comparison = pd.DataFrame(records)

# Overall summary
summary_rows = []
for col in predictor_cols:
    diff_col = f"{col}_abs_diff"
    if diff_col not in df_comparison.columns:
        continue
    diffs = df_comparison[diff_col].dropna()
    thresh = diffs.mean() + 2 * diffs.std()
    summary_rows.append({
        "feature": col,
        "mean_abs_diff": diffs.mean(),
        "median_abs_diff": diffs.median(),
        "max_abs_diff": diffs.max(),
        "std_abs_diff": diffs.std(),
        "threshold(μ+2σ)": thresh,
        "pct_above_threshold": 100.0 * (diffs > thresh).mean(),
    })

summary_df = pd.DataFrame(summary_rows)

print("Nearest-station comparison summary:")
print(summary_df.to_string(index=False))

print("\nOverall distance and feature-difference stats:")
print(df_comparison[["station_distance_km", "l2_diff", "max_abs_diff", "mean_abs_diff"]].describe())

# Optional: flag the most extreme grid cells
extreme = df_comparison.sort_values("l2_diff", ascending=False).head(20)
print("\nTop 20 largest anomaly-distance grid cells:")
print(extreme[["date", "cell_idx", "nearest_station", "station_distance_km", "l2_diff", "max_abs_diff"]])

Nearest-station comparison summary:
             feature  mean_abs_diff  median_abs_diff  max_abs_diff  std_abs_diff  threshold(μ+2σ)  pct_above_threshold
      temperature_2m       0.418585         0.282212      3.811218      0.440830         1.300245             5.511885
relative_humidity_2m       1.540334         1.078526     17.342949      1.540490         4.621314             4.690717
        dew_point_2m       0.440112         0.254487      4.384295      0.516221         1.472554             5.752854
      wind_speed_10m       0.542269         0.434615      4.369712      0.457624         1.457516             5.036964
  wind_direction_10m      12.457889         5.857372    315.642628     24.384100        61.226089             2.973517
    surface_pressure       7.919350         4.119551     57.861699      9.809749        27.538848             5.427662
       precipitation       0.568720         0.030128     36.896154      1.820709         4.210139             3.265955

Overall dis

In [13]:
# Dynamic graph dataset, Dynamic GCN-GRU model, training, and baseline/test MSE
# Paste into your notebook; assumes `graphs` (dict date_iso -> nx.DiGraph) and `grouped` (DataFrame with 'location_id','date','value') exist.

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from torch_geometric.nn import GCNConv
import math

# -----------------------
# Config
# -----------------------
WINDOW = 3                # past days -> predict next day
EMBED_DIM = 64
GCN_OUT = 64
GRU_HIDDEN = 64
EPOCHS = 40
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------
# Dates: first 2 months, then 80/20 split (time-based)
# -----------------------
all_dates = sorted([pd.to_datetime(d).date() for d in graphs.keys()])
start_date = all_dates[0]
end_2mo = start_date + relativedelta(months=2)
dates_2mo = [d for d in all_dates if d <= end_2mo]
if len(dates_2mo) < WINDOW + 1:
    raise RuntimeError("Not enough dates in the 2-month window for chosen WINDOW")

split_idx = int(len(dates_2mo) * 0.8)
train_dates = dates_2mo[:split_idx]
test_dates = dates_2mo[split_idx:]
print(f"2-month dates: {len(dates_2mo)} | Train dates: {len(train_dates)} | Test dates: {len(test_dates)}")

# -----------------------
# Common nodes across the 2-month window (fixed node ordering)
# -----------------------
node_sets = [set(graphs[d.isoformat()].nodes()) for d in dates_2mo]
common_nodes = sorted(list(set.intersection(*node_sets)))
if len(common_nodes) == 0:
    raise RuntimeError("No common nodes across selected dates; choose a different window or relax requirement.")
node_to_idx = {n: i for i, n in enumerate(common_nodes)}
N = len(common_nodes)
print(f"Using {N} common nodes")

# -----------------------
# Helpers: extract features & targets for a date
# -----------------------
def get_node_features_for_date(date):
    """Return features array (N x F) and pm25 targets (N,) with NaNs where missing."""
    G = graphs[date.isoformat()]
    # Determine feature length
    any_node = next(iter(G.nodes(data=True)))[1]
    F = len(any_node["features"])
    feats = np.full((N, F), np.nan, dtype=float)
    targets = np.full((N,), np.nan, dtype=float)
    for node, attrs in G.nodes(data=True):
        if node in node_to_idx:
            i = node_to_idx[node]
            feats[i] = np.asarray(attrs["features"], dtype=float)
    day_df = grouped[grouped["date"] == date].set_index("location_id")
    for node in common_nodes:
        i = node_to_idx[node]
        if node in day_df.index:
            val = day_df.loc[node, "value"]
            targets[i] = float(val) if not pd.isna(val) else np.nan
    return feats, targets

# -----------------------
# Build sliding-window samples
# -----------------------
def build_samples(dates_list):
    samples = []
    for idx in range(WINDOW, len(dates_list)):
        input_dates = dates_list[idx-WINDOW:idx]
        target_date = dates_list[idx]
        feat_stack = []
        skip = False
        for d in input_dates:
            f, _ = get_node_features_for_date(d)
            if np.all(np.isnan(f)):
                skip = True
                break
            feat_stack.append(f)
        if skip:
            continue
        X = np.stack(feat_stack, axis=0)   # T x N x F
        _, y = get_node_features_for_date(target_date)
        samples.append({"X": X, "dates": input_dates, "y": y, "target_date": target_date})
    return samples

train_samples = build_samples(train_dates)
test_samples = build_samples(test_dates)
print(f"Train samples: {len(train_samples)}, Test samples: {len(test_samples)}")
if len(train_samples) == 0:
    raise RuntimeError("No training samples constructed; check WINDOW and date availability.")

# -----------------------
# Build adjacency (edge_index, edge_weight) per date in the common node ordering
# -----------------------
def adjacency_from_graph(date):
    G = graphs[date.isoformat()]
    edge_pairs = []
    weight_list = []
    for u, v, attrs in G.edges(data=True):
        if u in node_to_idx and v in node_to_idx:
            ui, vi = node_to_idx[u], node_to_idx[v]
            edge_pairs.append([ui, vi])
            weight_list.append(float(attrs.get("weight", 1.0)))
    if len(edge_pairs) == 0:
        # identity self-loops
        ei = torch.tensor([[i for i in range(N)], [i for i in range(N)]], dtype=torch.long).to(DEVICE)
        ew = torch.ones(N, dtype=torch.float32).to(DEVICE)
    else:
        ei = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous().to(DEVICE)  # [2, E]
        ew = torch.tensor(weight_list, dtype=torch.float32).to(DEVICE)               # [E]
    return (ei, ew)

adj_cache = {d: adjacency_from_graph(d) for d in dates_2mo}

# -----------------------
# Model: per-timestep GCN -> GRU across timesteps -> per-node MLP head
# -----------------------
class DynamicGCNGRU(nn.Module):
    def __init__(self, in_feats, emb=EMBED_DIM, gru_hidden=GRU_HIDDEN):
        super().__init__()
        self.in_proj = nn.Linear(in_feats, emb)
        self.gcn = GCNConv(emb, emb)
        self.gru = nn.GRU(input_size=emb, hidden_size=gru_hidden, batch_first=False)
        self.head = nn.Sequential(nn.Linear(gru_hidden, gru_hidden//2), nn.ReLU(), nn.Linear(gru_hidden//2, 1))
    def forward(self, x_seq, adj_seq):
        # x_seq: T x N x F (torch)
        T, Nn, F = x_seq.shape
        x_seq = self.in_proj(x_seq)   # T x N x emb
        h_seq = []
        for t in range(T):
            x_t = x_seq[t]            # N x emb
            edge_index, edge_weight = adj_seq[t]
            x_g = self.gcn(x_t, edge_index, edge_weight)
            h_seq.append(x_g.unsqueeze(0))
        h_cat = torch.cat(h_seq, dim=0)   # T x N x emb
        out, _ = self.gru(h_cat)          # T x N x hidden
        final = out[-1]                   # N x hidden
        preds = self.head(final).squeeze(-1)  # N
        return preds

# -----------------------
# Training preparation
# -----------------------
in_F = train_samples[0]["X"].shape[2]
model = DynamicGCNGRU(in_feats=in_F).to(DEVICE)
opt = optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss(reduction="mean")

def sample_to_tensors(sample):
    X = torch.tensor(sample["X"], dtype=torch.float32).to(DEVICE)   # T x N x F
    adj_seq = [adj_cache[d] for d in sample["dates"]]
    y = torch.tensor(sample["y"], dtype=torch.float32).to(DEVICE)   # N
    return X, adj_seq, y

# Baseline: per-node mean from training targets (ignore NaNs)
all_train_targets = np.stack([s["y"] for s in train_samples], axis=0)  # S x N
node_means = np.nanmean(all_train_targets, axis=0)  # N
global_mean = np.nanmean(node_means)
node_means = np.where(np.isnan(node_means), global_mean, node_means)

# Baseline MSE on test set
test_targets = np.stack([s["y"] for s in test_samples], axis=0)  # S_test x N
mask = ~np.isnan(test_targets)
baseline_preds = np.tile(node_means, (test_targets.shape[0], 1))
baseline_mse = np.mean((test_targets[mask] - baseline_preds[mask])**2)
print(f"Baseline (train-node-mean) MSE on test set: {baseline_mse:.6f}")

# -----------------------
# Train
# -----------------------
model.train()
for epoch in range(1, EPOCHS+1):
    total_loss = 0.0
    cnt = 0
    for s in train_samples:
        X, adj_seq, y = sample_to_tensors(s)
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        preds = model(X, adj_seq)
        loss = loss_fn(preds[mask], y[mask])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
        cnt += 1
    avg_loss = total_loss / max(1, cnt)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{EPOCHS} | Train MSE: {avg_loss:.6f}")

# -----------------------
# Evaluate on test set
# -----------------------
model.eval()
preds_list = []
targets_list = []
with torch.no_grad():
    for s in test_samples:
        X, adj_seq, y = sample_to_tensors(s)
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        preds = model(X, adj_seq).cpu().numpy()
        preds_list.append(preds)
        targets_list.append(y.cpu().numpy())

if len(targets_list) == 0:
    raise RuntimeError("No test targets available for evaluation.")
preds_arr = np.stack(preds_list, axis=0)
targets_arr = np.stack(targets_list, axis=0)
mask = ~np.isnan(targets_arr)
test_mse = np.mean((preds_arr[mask] - targets_arr[mask])**2)
print(f"Test MSE (model): {test_mse:.6f}")
print(f"Baseline MSE (node-mean): {baseline_mse:.6f}")

2-month dates: 63 | Train dates: 50 | Test dates: 13
Using 26 common nodes
Train samples: 47, Test samples: 10
Baseline (train-node-mean) MSE on test set: 87.272113
Epoch 1/40 | Train MSE: 421.154456
Epoch 5/40 | Train MSE: 103.775770
Epoch 10/40 | Train MSE: 98.860607
Epoch 15/40 | Train MSE: 96.166728
Epoch 20/40 | Train MSE: 94.677613
Epoch 25/40 | Train MSE: 93.225477
Epoch 30/40 | Train MSE: 90.321442
Epoch 35/40 | Train MSE: 86.591998
Epoch 40/40 | Train MSE: 81.549716
Test MSE (model): 114.020584
Baseline MSE (node-mean): 87.272113


In [22]:
import copy
import math
import numpy as np
import torch

# ----------------------------------------------------------------------
# 0. Build daily anomaly statistics for predictors and target
# ----------------------------------------------------------------------
def compute_daily_anomaly_stats(grouped, predictor_cols, target_col="value"):
    daily_stats = {}
    for d in sorted(grouped["date"].unique()):
        date_iso = d.isoformat() if hasattr(d, "isoformat") else str(d)
        day = grouped[grouped["date"] == d]

        raw_feats = day[predictor_cols].astype(float).values
        feat_mean = np.nanmean(raw_feats, axis=0)
        feat_std = np.nanstd(raw_feats, axis=0)
        feat_std[feat_std < 1e-6] = 1.0

        raw_target = day[target_col].astype(float).values
        target_mean = np.nanmean(raw_target)
        target_std = np.nanstd(raw_target)
        if np.isnan(target_mean):
            target_mean = 0.0
        if target_std < 1e-6:
            target_std = 1.0

        daily_stats[date_iso] = {
            "feat_mean": feat_mean,
            "feat_std": feat_std,
            "target_mean": target_mean,
            "target_std": target_std,
        }
    return daily_stats

daily_stats = compute_daily_anomaly_stats(grouped, predictor_cols)


# ----------------------------------------------------------------------
# 1. Candidate feature helpers
# ----------------------------------------------------------------------
def _build_raw_candidate_vector(raw_entry, predictor_cols):
    raw_vec = np.full((len(predictor_cols),), np.nan, dtype=float)
    if raw_entry is None:
        return raw_vec

    for i, col in enumerate(predictor_cols):
        if col in raw_entry and raw_entry[col] is not None:
            try:
                raw_vec[i] = float(raw_entry[col])
            except Exception:
                raw_vec[i] = np.nan
    return raw_vec


def _align_vector_to_length(vec, length):
    vec = np.asarray(vec, dtype=float)
    if vec.size == length:
        return vec
    if vec.size < length:
        return np.concatenate([vec, np.full(length - vec.size, np.nan, dtype=float)])
    return vec[:length]


def _get_candidate_day_entry(cell_idx, date_iso):
    """Return (lon, lat, raw_entry, anomaly_features)."""
    raw_entry = None
    cand_raw_vec = None
    lon = None
    lat = None

    if "data_by_day" in globals() and date_iso in data_by_day and int(cell_idx) in data_by_day[date_iso]:
        raw_entry = data_by_day[date_iso][int(cell_idx)]

    if "grid_features_by_day" in globals() and date_iso in grid_features_by_day and int(cell_idx) in grid_features_by_day[date_iso]:
        grid_entry = grid_features_by_day[date_iso][int(cell_idx)]
        cand_raw_vec = np.asarray(grid_entry.get("features", np.zeros(0)), dtype=float)
        lon = float(grid_entry["lon"])
        lat = float(grid_entry["lat"])

    if raw_entry is not None:
        lon = lon if lon is not None else float(raw_entry.get("lon", np.nan))
        lat = lat if lat is not None else float(raw_entry.get("lat", np.nan))
        raw_vec = _build_raw_candidate_vector(raw_entry, predictor_cols)
        if cand_raw_vec is None or np.all(np.isnan(cand_raw_vec)):
            cand_raw_vec = raw_vec

    if cand_raw_vec is None:
        return None, None, raw_entry, None

    cand_raw_vec = _align_vector_to_length(cand_raw_vec, len(predictor_cols))
    return lon, lat, raw_entry, cand_raw_vec


def _candidate_anomaly_features(cell_idx, date_iso, feature_dim):
    stats = daily_stats.get(date_iso)
    if stats is None:
        return np.zeros((feature_dim,), dtype=float)

    cand_lon, cand_lat, cand_raw, cand_raw_vec = _get_candidate_day_entry(cell_idx, date_iso)
    if cand_raw_vec is None:
        return np.zeros((feature_dim,), dtype=float)

    cand_raw_vec = _align_vector_to_length(cand_raw_vec, feature_dim)
    cand_raw_vec = np.where(np.isnan(cand_raw_vec), stats["feat_mean"], cand_raw_vec)
    return cand_raw_vec - stats["feat_mean"]


def _anomaly_target_for_date(date_iso, raw_values):
    stats = daily_stats.get(date_iso)
    if stats is None:
        return raw_values
    return (raw_values - stats["target_mean"]) / stats["target_std"]


# ----------------------------------------------------------------------
# 2. Existing station wind helper
# ----------------------------------------------------------------------
def _get_station_wind_for_date(date_obj, station_id):
    try:
        day_df = grouped[grouped["date"] == date_obj].set_index("location_id")
        if station_id in day_df.index:
            row = day_df.loc[station_id]
            wspeed = float(row["wind_speed_10m"]) if not pd.isna(row["wind_speed_10m"]) else 0.0
            wdir = float(row["wind_direction_10m"]) if not pd.isna(row["wind_direction_10m"]) else 0.0
            return wspeed, wdir
    except Exception:
        pass
    return 0.0, 0.0


# ----------------------------------------------------------------------
# 3. Augmented adjacency builder
# ----------------------------------------------------------------------
def build_augmented_adj_and_edges_for_date(date_obj, cell_idx, cand_lon, cand_lat, cand_raw):
    G = graphs[date_obj.isoformat()]
    edge_pairs = []
    weight_list = []

    for u, v, attrs in G.edges(data=True):
        if u in node_to_idx and v in node_to_idx:
            ui, vi = node_to_idx[u], node_to_idx[v]
            edge_pairs.append([ui, vi])
            weight_list.append(float(attrs.get("weight", 1.0)))

    cidx = N
    for node in common_nodes:
        ni = node_to_idx[node]
        n_attr = G.nodes[node]
        s_lat = float(n_attr["station_lat"])
        s_lon = float(n_attr["station_lon"])
        dist = haversine_km(cand_lat, cand_lon, s_lat, s_lon)
        if dist <= DIST_THRESHOLD_KM:
            cand_wind_speed = 0.0
            cand_wind_dir = None
            if cand_raw is not None:
                cand_wind_speed = float(
                    cand_raw.get("windspeed_10m_mean")
                    or cand_raw.get("wind_speed_10m")
                    or 0.0
                )
                cand_wind_dir = (
                    cand_raw.get("winddirection_10m_dominant")
                    or cand_raw.get("wind_direction_10m")
                    or None
                )
                if cand_wind_dir is not None:
                    cand_wind_dir = float(cand_wind_dir)

            bearing_c_to_s = bearing_deg(cand_lat, cand_lon, s_lat, s_lon)
            if cand_wind_dir is None:
                cand_score = 0.0
            else:
                diff = angle_diff_deg(cand_wind_dir, bearing_c_to_s)
                cand_score = max(math.cos(math.radians(diff)), 0.0) * cand_wind_speed

            if cand_score > 0:
                edge_pairs.append([cidx, ni])
                weight_list.append(float(cand_score))

            station_wind_speed, station_wind_dir = _get_station_wind_for_date(date_obj, node)
            bearing_s_to_c = bearing_deg(s_lat, s_lon, cand_lat, cand_lon)
            diff2 = angle_diff_deg(station_wind_dir, bearing_s_to_c)
            station_score = max(math.cos(math.radians(diff2)), 0.0) * station_wind_speed
            if station_score > 0:
                edge_pairs.append([ni, cidx])
                weight_list.append(float(station_score))

    if all(pair[0] != cidx for pair in edge_pairs):
        cand_ws = 0.0
        if cand_raw is not None:
            cand_ws = float(
                cand_raw.get("windspeed_10m_mean")
                or cand_raw.get("wind_speed_10m")
                or 0.0
            )
        if cand_ws == 0.0:
            edge_pairs.append([cidx, cidx])
            weight_list.append(1.0)

    if len(edge_pairs) == 0:
        ei = torch.tensor([[i for i in range(N)], [i for i in range(N)]], dtype=torch.long).to(DEVICE)
        ew = torch.ones(N, dtype=torch.float32).to(DEVICE)
        return ei, ew

    ei = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous().to(DEVICE)
    ew = torch.tensor(weight_list, dtype=torch.float32).to(DEVICE)
    return ei, ew


# ----------------------------------------------------------------------
# 4. Candidate evaluation with daily anomaly features
# ----------------------------------------------------------------------
def evaluate_candidate_cell(cell_idx, verbose=False):
    model.eval()
    orig_preds_list = []
    orig_targets_list = []
    with torch.no_grad():
        for s in test_samples:
            X, adj_seq, y = sample_to_tensors(s)
            mask = ~torch.isnan(y)
            if mask.sum() == 0:
                continue
            preds = model(X, adj_seq).cpu().numpy()
            orig_preds_list.append(preds)
            orig_targets_list.append(y.cpu().numpy())

    if len(orig_targets_list) == 0:
        raise RuntimeError("No test targets available.")
    orig_preds = np.stack(orig_preds_list, axis=0)
    orig_targets = np.stack(orig_targets_list, axis=0)
    orig_mask = ~np.isnan(orig_targets)
    original_mse_recalc = float(np.mean((orig_preds[orig_mask] - orig_targets[orig_mask]) ** 2))

    aug_preds_list = []
    aug_targets_list = []
    with torch.no_grad():
        for s in test_samples:
            T, _, F = s["X"].shape
            X_aug = []
            adj_seq_aug = []
            for d_idx, date_obj in enumerate(s["dates"]):
                date_iso = date_obj.isoformat()
                X_t = s["X"][d_idx]

                cand_lon, cand_lat, cand_raw, _ = _get_candidate_day_entry(cell_idx, date_iso)
                cand_feat = _candidate_anomaly_features(cell_idx, date_iso, F)
                X_t_aug = np.vstack([X_t, cand_feat])
                X_aug.append(X_t_aug)

                if cand_lon is None or cand_lat is None:
                    orig_ei, orig_ew = adj_cache[date_obj]
                    adj_seq_aug.append((orig_ei, orig_ew))
                else:
                    ei, ew = build_augmented_adj_and_edges_for_date(
                        date_obj, cell_idx, cand_lon, cand_lat, cand_raw
                    )
                    adj_seq_aug.append((ei, ew))

            X_aug_t = torch.tensor(np.stack(X_aug, axis=0), dtype=torch.float32).to(DEVICE)
            try:
                preds_aug_all = model(X_aug_t, adj_seq_aug).cpu().numpy()
            except Exception as e:
                if verbose:
                    print(f"Model inference failed for candidate {cell_idx} at sample: {e}")
                continue

            preds_existing = preds_aug_all[:N]
            aug_preds_list.append(preds_existing)
            aug_targets_list.append(s["y"])

    if len(aug_targets_list) == 0:
        raise RuntimeError("No augmented test targets available.")

    aug_preds_arr = np.stack(aug_preds_list, axis=0)
    aug_targets_arr = np.stack(aug_targets_list, axis=0)
    aug_mask = ~np.isnan(aug_targets_arr)
    augmented_mse = float(np.mean((aug_preds_arr[aug_mask] - aug_targets_arr[aug_mask]) ** 2))

    return {
        "cell_idx": int(cell_idx),
        "original_test_mse": original_mse_recalc,
        "augmented_test_mse": augmented_mse,
        "delta_mse": augmented_mse - original_mse_recalc,
    }


def evaluate_all_grid_cells(cell_indices=None, top_k=20, verbose=False):
    if cell_indices is None:
        if "grid_centers" in globals():
            cell_indices = list(range(len(grid_centers)))
        elif "data_by_day" in globals():
            sample_date = next(iter(data_by_day.keys()))
            cell_indices = sorted([int(k) for k in data_by_day[sample_date].keys()])
        else:
            raise RuntimeError("No candidate grid indices available.")
    results = []
    for idx in cell_indices:
        try:
            res = evaluate_candidate_cell(idx, verbose=verbose)
            results.append(res)
            if verbose:
                print(f"Cell {idx}: delta_mse={res['delta_mse']:.6f}")
        except Exception as e:
            if verbose:
                print(f"Cell {idx} failed: {e}")
            continue
    results_sorted = sorted(results, key=lambda r: r["delta_mse"])
    return results_sorted[:top_k], results_sorted


# ----------------------------------------------------------------------
# 5. Run evaluation
# ----------------------------------------------------------------------
top20, all_results = evaluate_all_grid_cells(top_k=5, verbose=True)
for r in top20:
    idx = r["cell_idx"]
    date_example = next(iter(grid_features_by_day.keys()))
    lonlat = grid_features_by_day[date_example].get(idx, None)
    print(idx, (lonlat["lon"], lonlat["lat"]) if lonlat is not None else None, r)

Cell 0: delta_mse=0.000000
Cell 1: delta_mse=0.000000
Cell 2: delta_mse=0.000000
Cell 3: delta_mse=0.000000
Cell 4: delta_mse=0.000000
Cell 5: delta_mse=0.000000
Cell 6: delta_mse=0.000000
Cell 7: delta_mse=0.000000
Cell 8: delta_mse=0.000000
Cell 9: delta_mse=-1.605451
Cell 10: delta_mse=-0.231594
Cell 11: delta_mse=0.502085
Cell 12: delta_mse=0.739576
Cell 13: delta_mse=0.000000
Cell 14: delta_mse=0.000000
Cell 15: delta_mse=-1.380515
Cell 16: delta_mse=-1.768163
Cell 17: delta_mse=-1.928174
Cell 18: delta_mse=-1.266690
Cell 19: delta_mse=0.000000
Cell 20: delta_mse=0.000000
Cell 21: delta_mse=0.000000
Cell 22: delta_mse=0.045827
Cell 23: delta_mse=0.201996
Cell 24: delta_mse=0.735024
Cell 25: delta_mse=0.368933
Cell 26: delta_mse=0.000000
Cell 27: delta_mse=-0.942117
Cell 28: delta_mse=-0.492862
Cell 29: delta_mse=-2.069651
Cell 30: delta_mse=-8.206609
Cell 31: delta_mse=-6.042078
Cell 32: delta_mse=-5.443756
Cell 33: delta_mse=-1.172866
Cell 34: delta_mse=0.000000
Cell 35: delta_ms

In [23]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------
# Params
# -----------------------
TOP_K = 5
EPOCHS_AUG = 30
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DIST_THRESHOLD_KM = globals().get("DIST_THRESHOLD_KM", 5.0)

# -----------------------
# 1) Ranking
# -----------------------
if "results_sorted" in globals():
    ranked = results_sorted
elif "all_results" in globals():
    ranked = sorted(all_results, key=lambda r: r["delta_mse"])
elif "evaluate_all_grid_cells" in globals():
    _, all_results = evaluate_all_grid_cells(top_k=None, verbose=False)
    ranked = sorted(all_results, key=lambda r: r["delta_mse"])
else:
    raise RuntimeError(
        "No candidate ranking found. Run evaluate_all_grid_cells(...) first or provide `all_results`."
    )

topk = [int(r["cell_idx"]) for r in ranked[:TOP_K]]
print("Top-K candidate indices:", topk)

# -----------------------
# 2) Candidate info helper
# -----------------------
def get_candidate_info_for_date(cell_idx, date_iso, F_target):
    lon = lat = None
    raw = None
    feat_raw = np.zeros(0)
    if (
        "grid_features_by_day" in globals()
        and date_iso in grid_features_by_day
        and int(cell_idx) in grid_features_by_day[date_iso]
    ):
        entry = grid_features_by_day[date_iso][int(cell_idx)]
        lon = float(entry["lon"])
        lat = float(entry["lat"])
        feat_raw = np.asarray(entry.get("features", np.zeros(0)), dtype=float)
    if (
        "data_by_day" in globals()
        and date_iso in data_by_day
        and int(cell_idx) in data_by_day[date_iso]
    ):
        raw = data_by_day[date_iso][int(cell_idx)]
        if lon is None:
            lon = float(raw.get("lon", np.nan))
            lat = float(raw.get("lat", np.nan))
    if feat_raw.size == 0:
        feat = np.zeros(F_target, dtype=float)
    else:
        if feat_raw.size < F_target:
            feat = np.concatenate([feat_raw, np.zeros(F_target - feat_raw.size, dtype=float)])
        else:
            feat = feat_raw[:F_target]
    return lon, lat, raw, feat


def _align_vector_to_length(vec, length):
    vec = np.asarray(vec, dtype=float)
    if vec.size == length:
        return vec
    if vec.size < length:
        return np.concatenate([vec, np.full(length - vec.size, np.nan, dtype=float)])
    return vec[:length]


def _daily_predictor_baseline(date_obj, F):
    day = grouped[grouped["date"] == date_obj]
    if day.empty:
        return np.zeros((F,), dtype=float)
    raw_feats = day[predictor_cols].astype(float).values
    baseline = np.nanmean(raw_feats, axis=0)
    return _align_vector_to_length(baseline, F)


def _candidate_anomaly_features(cell_idx, date_obj, F):
    date_iso = date_obj.isoformat()
    _, _, _, feat_vec = get_candidate_info_for_date(cell_idx, date_iso, F)
    feat_vec = _align_vector_to_length(feat_vec, F)
    baseline = _daily_predictor_baseline(date_obj, F)
    return np.where(np.isnan(feat_vec), baseline, feat_vec) - baseline


def _daily_target_stats(date_obj):
    day = grouped[grouped["date"] == date_obj]
    values = day["value"].astype(float).values
    if values.size == 0:
        return 0.0, 1.0
    mean = np.nanmean(values)
    std = np.nanstd(values)
    if np.isnan(mean):
        mean = 0.0
    if std < 1e-6:
        std = 1.0
    return mean, std


def _anomaly_targets(raw_targets, date_obj):
    mean, std = _daily_target_stats(date_obj)
    return (raw_targets - mean) / std, mean, std


# -----------------------
# 3) Build augmented graphs
# -----------------------
graphs_aug = {}
for d in dates_2mo:
    date_iso = d.isoformat()
    G = copy.deepcopy(graphs[date_iso])
    any_node = next(iter(G.nodes(data=True)))[1]
    F = len(any_node["features"])
    for cell_idx in topk:
        cand_node_id = f"CAND_{cell_idx}"
        lon, lat, raw, _ = get_candidate_info_for_date(cell_idx, date_iso, F)
        if lon is None or lat is None:
            lon = float("nan")
            lat = float("nan")
        cand_feat = _candidate_anomaly_features(cell_idx, d, F)

        G.add_node(
            cand_node_id,
            station_lat=float(lat) if not np.isnan(lat) else float("nan"),
            station_lon=float(lon) if not np.isnan(lon) else float("nan"),
            features=cand_feat,
            feature_names=any_node.get("feature_names", None),
        )

        for node, attrs in list(G.nodes(data=True)):
            if node == cand_node_id:
                continue
            try:
                s_lat = float(attrs["station_lat"])
                s_lon = float(attrs["station_lon"])
            except Exception:
                continue
            if np.isnan(s_lat) or np.isnan(s_lon) or np.isnan(lat) or np.isnan(lon):
                continue
            dist = haversine_km(lat, lon, s_lat, s_lon)
            if dist <= DIST_THRESHOLD_KM:
                cand_ws = 0.0
                cand_wd = None
                if raw is not None:
                    cand_ws = float(
                        raw.get("windspeed_10m_mean")
                        or raw.get("wind_speed_10m")
                        or 0.0
                    )
                    cand_wd = (
                        raw.get("winddirection_10m_dominant")
                        or raw.get("wind_direction_10m")
                        or None
                    )
                    if cand_wd is not None:
                        cand_wd = float(cand_wd)

                if cand_wd is not None and cand_ws > 0:
                    bearing = bearing_deg(lat, lon, s_lat, s_lon)
                    diff = angle_diff_deg(cand_wd, bearing)
                    score = max(np.cos(np.radians(diff)), 0.0) * cand_ws
                    if score > 0:
                        G.add_edge(
                            cand_node_id,
                            node,
                            weight=float(score),
                            distance_km=dist,
                        )

                try:
                    day_df = grouped[grouped["date"] == d].set_index("location_id")
                    if node in day_df.index:
                        st_row = day_df.loc[node]
                        st_ws = (
                            float(st_row["wind_speed_10m"])
                            if not pd.isna(st_row["wind_speed_10m"])
                            else 0.0
                        )
                        st_wd = (
                            float(st_row["wind_direction_10m"])
                            if not pd.isna(st_row["wind_direction_10m"])
                            else None
                        )
                    else:
                        st_ws = 0.0
                        st_wd = None
                except Exception:
                    st_ws = 0.0
                    st_wd = None

                if st_wd is not None and st_ws > 0:
                    bearing2 = bearing_deg(s_lat, s_lon, lat, lon)
                    diff2 = angle_diff_deg(st_wd, bearing2)
                    score2 = max(np.cos(np.radians(diff2)), 0.0) * st_ws
                    if score2 > 0:
                        G.add_edge(
                            node,
                            cand_node_id,
                            weight=float(score2),
                            distance_km=dist,
                        )

        cand_ws_check = 0.0
        if raw is not None:
            cand_ws_check = float(
                raw.get("windspeed_10m_mean")
                or raw.get("wind_speed_10m")
                or 0.0
            )
        if cand_ws_check == 0.0 and not G.has_edge(cand_node_id, cand_node_id):
            G.add_edge(cand_node_id, cand_node_id, weight=1.0, distance_km=0.0)

    graphs_aug[date_iso] = G

print("Augmented graphs built for dates:", len(graphs_aug))

# -----------------------
# 4) Build common_aug robustly
# -----------------------
candidate_ids = [f"CAND_{idx}" for idx in topk]
orig_common = common_nodes

orig_nodes_in_all = set(orig_common)
for d in dates_2mo:
    nodes = set(graphs_aug[d.isoformat()].nodes())
    orig_nodes_in_all &= {n for n in nodes if n in orig_common}

common_aug = sorted(list(orig_nodes_in_all), key=lambda x: str(x))
for cid in candidate_ids:
    if cid not in common_aug:
        common_aug.append(cid)

node_to_idx_aug = {n: i for i, n in enumerate(common_aug)}
N_aug = len(common_aug)
print(
    f"Original common nodes kept: {len(common_aug) - len(candidate_ids)} | "
    f"Total nodes (with candidates): {N_aug}"
)

# -----------------------
# 5) Build augmented features + target helper
# -----------------------
def get_aug_features_for_date(date_obj):
    G = graphs_aug[date_obj.isoformat()]
    any_node = next(iter(G.nodes(data=True)))[1]
    F = len(any_node["features"])
    feats = np.full((N_aug, F), np.nan, dtype=float)
    targets = np.full((N_aug,), np.nan, dtype=float)

    for node, attrs in G.nodes(data=True):
        if node in node_to_idx_aug:
            i = node_to_idx_aug[node]
            feats[i] = np.asarray(attrs.get("features", np.zeros(F)), dtype=float)

    day_df = grouped[grouped["date"] == date_obj].set_index("location_id")
    for node in common_aug:
        node_str = str(node)
        if node_str.startswith("CAND_"):
            continue
        i = node_to_idx_aug[node]
        key = node
        if key not in day_df.index:
            try:
                key_alt = int(node_str)
                if key_alt in day_df.index:
                    key = key_alt
            except Exception:
                pass
        if key in day_df.index:
            val = day_df.loc[key, "value"]
            targets[i] = float(val) if not pd.isna(val) else np.nan

    return feats, targets


def build_aug_samples(dates_list):
    samples = []
    for idx in range(WINDOW, len(dates_list)):
        input_dates = dates_list[idx - WINDOW : idx]
        target_date = dates_list[idx]
        feat_stack = []
        skip = False
        for d in input_dates:
            f, _ = get_aug_features_for_date(d)
            if np.all(np.isnan(f)):
                skip = True
                break
            feat_stack.append(np.where(np.isnan(f), 0.0, f))
        if skip:
            continue

        X = np.stack(feat_stack, axis=0)  # T x N_aug x F
        _, y_raw = get_aug_features_for_date(target_date)
        y_norm, mean_t, std_t = _anomaly_targets(y_raw, target_date)
        samples.append(
            {
                "X": X,
                "dates": input_dates,
                "y": y_norm,
                "y_raw": y_raw,
                "target_mean": mean_t,
                "target_std": std_t,
                "target_date": target_date,
            }
        )
    return samples


train_samples_aug = build_aug_samples(train_dates)
test_samples_aug = build_aug_samples(test_dates)
print(
    f"Augmented Train samples: {len(train_samples_aug)}, "
    f"Augmented Test samples: {len(test_samples_aug)}"
)
if len(train_samples_aug) == 0:
    raise RuntimeError(
        "No augmented training samples constructed; check WINDOW and availability."
    )

# -----------------------
# 6) Adjacency cache for augmented graphs
# -----------------------
def adjacency_from_graph_aug(date_obj):
    G = graphs_aug[date_obj.isoformat()]
    edge_pairs = []
    weight_list = []
    for u, v, attrs in G.edges(data=True):
        if u in node_to_idx_aug and v in node_to_idx_aug:
            ui, vi = node_to_idx_aug[u], node_to_idx_aug[v]
            edge_pairs.append([ui, vi])
            weight_list.append(float(attrs.get("weight", 1.0)))
    if len(edge_pairs) == 0:
        ei = torch.tensor(
            [[i for i in range(N_aug)], [i for i in range(N_aug)]],
            dtype=torch.long,
        ).to(DEVICE)
        ew = torch.ones(N_aug, dtype=torch.float32).to(DEVICE)
    else:
        ei = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous().to(DEVICE)
        ew = torch.tensor(weight_list, dtype=torch.float32).to(DEVICE)
    return ei, ew


adj_cache_aug = {d: adjacency_from_graph_aug(d) for d in dates_2mo}

# -----------------------
# 7) Train augmented model
# -----------------------
in_F = train_samples_aug[0]["X"].shape[2]
model_aug = DynamicGCNGRU(in_feats=in_F).to(DEVICE)
opt = optim.Adam(model_aug.parameters(), lr=LR)
loss_fn = nn.MSELoss(reduction="mean")


def sample_to_tensors_aug(sample):
    X = torch.tensor(sample["X"], dtype=torch.float32).to(DEVICE)
    adj_seq = [adj_cache_aug[d] for d in sample["dates"]]
    y = torch.tensor(sample["y"], dtype=torch.float32).to(DEVICE)
    return X, adj_seq, y


# baseline in raw units using train targets
all_train_targets_raw = np.stack([s["y_raw"] for s in train_samples_aug], axis=0)
node_means_aug = np.nanmean(all_train_targets_raw, axis=0)
global_mean_aug = np.nanmean(node_means_aug)
node_means_aug = np.where(np.isnan(node_means_aug), global_mean_aug, node_means_aug)

test_targets_raw = np.stack([s["y_raw"] for s in test_samples_aug], axis=0)
mask_raw = ~np.isnan(test_targets_raw)
baseline_preds_aug = np.tile(node_means_aug, (test_targets_raw.shape[0], 1))
baseline_mse_aug = np.mean((test_targets_raw[mask_raw] - baseline_preds_aug[mask_raw]) ** 2)
print(f"Augmented baseline (train node mean) MSE on test set: {baseline_mse_aug:.6f}")

model_aug.train()
for epoch in range(1, EPOCHS_AUG + 1):
    total_loss = 0.0
    cnt = 0
    for s in train_samples_aug:
        X, adj_seq, y = sample_to_tensors_aug(s)
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        preds = model_aug(X, adj_seq)
        loss = loss_fn(preds[mask], y[mask])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
        cnt += 1
    avg_loss = total_loss / max(1, cnt)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{EPOCHS_AUG} | Train MSE: {avg_loss:.6f}")

# -----------------------
# 8) Evaluate on original nodes only
# -----------------------
model_aug.eval()
preds_list_aug = []
targets_list_aug = []
with torch.no_grad():
    for s in test_samples_aug:
        X, adj_seq, _ = sample_to_tensors_aug(s)
        preds = model_aug(X, adj_seq).cpu().numpy()
        preds_raw = preds * s["target_std"] + s["target_mean"]

        orig_pred = []
        orig_target = []
        for node in orig_common:
            if node in node_to_idx_aug:
                i = node_to_idx_aug[node]
                orig_pred.append(preds_raw[i])
                orig_target.append(s["y_raw"][i])
        if len(orig_target) == 0:
            continue
        preds_list_aug.append(np.array(orig_pred))
        targets_list_aug.append(np.array(orig_target))

if len(targets_list_aug) == 0:
    raise RuntimeError("No evaluation targets after augmentation.")

preds_arr_aug = np.stack(preds_list_aug, axis=0)
targets_arr_aug = np.stack(targets_list_aug, axis=0)
mask_eval = ~np.isnan(targets_arr_aug)
aug_test_mse_on_original = float(
    np.mean((preds_arr_aug[mask_eval] - targets_arr_aug[mask_eval]) ** 2)
)

print(f"Augmented Model Test MSE (on existing original nodes): {aug_test_mse_on_original:.6f}")
print(f"Original Model Test MSE (before augmentation): {globals().get('test_mse', 'unknown')}")

print("\nSummary:")
print(f"Baseline (original node-mean) MSE before augmentation: {globals().get('baseline_mse', 'unknown')}")
print(f"Model Test MSE before augmentation: {globals().get('test_mse', 'unknown')}")
print(f"Baseline (augmented node-mean) MSE on test set: {baseline_mse_aug:.6f}")
print(f"Augmented Model Test MSE on original nodes: {aug_test_mse_on_original:.6f}")

Top-K candidate indices: [30, 85, 31, 32, 45]
Augmented graphs built for dates: 63
Original common nodes kept: 26 | Total nodes (with candidates): 31
Augmented Train samples: 47, Augmented Test samples: 10
Augmented baseline (train node mean) MSE on test set: 87.272113
Epoch 1/30 | Train MSE: 0.968267


/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_833/4274592772.py:374: RuntimeWarning: Mean of empty slice
  node_means_aug = np.nanmean(all_train_targets_raw, axis=0)


Epoch 5/30 | Train MSE: 0.834786
Epoch 10/30 | Train MSE: 0.742404
Epoch 15/30 | Train MSE: 0.692804
Epoch 20/30 | Train MSE: 0.607015
Epoch 25/30 | Train MSE: 0.533675
Epoch 30/30 | Train MSE: 0.477487
Augmented Model Test MSE (on existing original nodes): 10.831611
Original Model Test MSE (before augmentation): 114.02058410644531

Summary:
Baseline (original node-mean) MSE before augmentation: 87.27211346028609
Model Test MSE before augmentation: 114.02058410644531
Baseline (augmented node-mean) MSE on test set: 87.272113
Augmented Model Test MSE on original nodes: 10.831611


In [24]:
import numpy as np

# 1) Baseline on ORIGINAL nodes only
orig_indices = [node_to_idx_aug[n] for n in common_aug if not str(n).startswith("CAND_")]
all_train_targets_raw = np.stack([s["y_raw"] for s in train_samples_aug], axis=0)
train_orig = all_train_targets_raw[:, orig_indices]

node_means_orig = np.nanmean(train_orig, axis=0)
global_mean_orig = np.nanmean(node_means_orig)
node_means_orig = np.where(np.isnan(node_means_orig), global_mean_orig, node_means_orig)

test_targets_raw = np.stack([s["y_raw"] for s in test_samples_aug], axis=0)
test_orig = test_targets_raw[:, orig_indices]
mask_orig = ~np.isnan(test_orig)
baseline_preds_orig = np.tile(node_means_orig, (test_orig.shape[0], 1))
baseline_orig_mse = np.mean((test_orig[mask_orig] - baseline_preds_orig[mask_orig]) ** 2)

print("Baseline on original nodes only:")
print("  original node count:", len(orig_indices))
print("  baseline_orig_mse:", baseline_orig_mse)
print("  baseline_aug (current):", globals().get("baseline_mse_aug", np.nan))
print("  baseline_orig - baseline_aug:", baseline_orig_mse - globals().get("baseline_mse_aug", np.nan))
print()

# 1b) Check whether candidate positions are all NaN in train targets
nan_target_counts = np.sum(np.isnan(all_train_targets_raw), axis=0)
all_nan_positions = [i for i, c in enumerate(nan_target_counts) if c == all_train_targets_raw.shape[0]]
print("Candidate positions with ALL-NaN train targets:", len(all_nan_positions), all_nan_positions[:20])
print("Total augmented nodes:", N_aug)
print()

# 2) Target scaling consistency
print("Target scaling checks:")
for i, s in enumerate(train_samples_aug[:5]):
    y = s["y"]
    mean_y = np.nanmean(y)
    std_y = np.nanstd(y)
    print(f"  train_sample {i}: mean(y)={mean_y:.6e}, std(y)={std_y:.6e}")

for i, s in enumerate(test_samples_aug[:5]):
    y_norm = s["y"]
    y_raw = s["y_raw"]
    mean = s["target_mean"]
    std = s["target_std"]
    y_inv = y_norm * std + mean
    diff = np.nanmax(np.abs(y_inv - y_raw))
    print(f"  test_sample {i}: inverse-transform max abs diff = {diff:.6e}")

print()

# 2b) Check that original node targets are derived from daily mean
for i, s in enumerate(test_samples_aug[:5]):
    y_norm = s["y"]
    mask = ~np.isnan(y_norm)
    if mask.sum() > 0:
        print(
            f"  test_sample {i}: normalized target mean = {np.nanmean(y_norm[mask]):.6e}, "
            f"std = {np.nanstd(y_norm[mask]):.6e}"
        )

print()

# 3) Padding check for candidate feature vectors
F = train_samples_aug[0]["X"].shape[2]
raw_lengths = []
missing_dates = []
for d in dates_2mo:
    date_iso = d.isoformat()
    for cell_idx in topk:
        if (
            "grid_features_by_day" in globals()
            and date_iso in grid_features_by_day
            and int(cell_idx) in grid_features_by_day[date_iso]
        ):
            vec = np.asarray(
                grid_features_by_day[date_iso][int(cell_idx)].get("features", np.zeros(0)),
                dtype=float,
            )
            raw_lengths.append(vec.size)
        else:
            missing_dates.append((date_iso, cell_idx))

print("Candidate feature raw lengths:", sorted(set(raw_lengths)))
print("Expected feature dimension:", F)
print("Missing candidate raw feature entries:", len(missing_dates), "examples:", missing_dates[:10])

# 3b) Check whether any candidate features were actually padded inside get_candidate_info_for_date
pad_counts = 0
for d in dates_2mo:
    date_iso = d.isoformat()
    for cell_idx in topk:
        lon, lat, raw, feat = get_candidate_info_for_date(cell_idx, date_iso, F)
        if feat.size == F:
            if "grid_features_by_day" in globals() and date_iso in grid_features_by_day and int(cell_idx) in grid_features_by_day[date_iso]:
                raw_vec = np.asarray(
                    grid_features_by_day[date_iso][int(cell_idx)].get("features", np.zeros(0)),
                    dtype=float,
                )
                if raw_vec.size != F:
                    pad_counts += 1
        else:
            pad_counts += 1
print("Candidate feature rows requiring pad/truncation:", pad_counts)

Baseline on original nodes only:
  original node count: 26
  baseline_orig_mse: 87.27211346028609
  baseline_aug (current): 87.27211346028609
  baseline_orig - baseline_aug: 0.0

Candidate positions with ALL-NaN train targets: 5 [26, 27, 28, 29, 30]
Total augmented nodes: 31

Target scaling checks:
  train_sample 0: mean(y)=5.689893e-16, std(y)=1.000000e+00
  train_sample 1: mean(y)=4.782499e-16, std(y)=1.000000e+00
  train_sample 2: mean(y)=-1.098480e-15, std(y)=1.000000e+00
  train_sample 3: mean(y)=-6.095551e-16, std(y)=1.000000e+00
  train_sample 4: mean(y)=2.391250e-16, std(y)=1.000000e+00
  test_sample 0: inverse-transform max abs diff = 0.000000e+00
  test_sample 1: inverse-transform max abs diff = 0.000000e+00
  test_sample 2: inverse-transform max abs diff = 0.000000e+00
  test_sample 3: inverse-transform max abs diff = 0.000000e+00
  test_sample 4: inverse-transform max abs diff = 0.000000e+00

  test_sample 0: normalized target mean = 4.077935e-16, std = 1.000000e+00
  test_

In [25]:
import numpy as np
import torch

# 1) confirm same test dates between original and augmented sample sets
assert len(test_samples) == len(test_samples_aug), "Test sample counts differ"
for s_orig, s_aug in zip(test_samples, test_samples_aug):
    assert s_orig["target_date"] == s_aug["target_date"], (
        s_orig["target_date"], s_aug["target_date"]
    )
print("Test sample dates align exactly.")

# 2) recompute original model raw MSE on the same test dates
orig_preds = []
orig_targets = []
model.eval()
with torch.no_grad():
    for s in test_samples:
        X, adj_seq, y = sample_to_tensors(s)
        preds = model(X, adj_seq).cpu().numpy()
        orig_preds.append(preds)
        orig_targets.append(y.cpu().numpy())

orig_preds = np.stack(orig_preds, axis=0)
orig_targets = np.stack(orig_targets, axis=0)
mask = ~np.isnan(orig_targets)
orig_model_mse_raw = float(np.mean((orig_preds[mask] - orig_targets[mask]) ** 2))
print("Original model raw test MSE:", orig_model_mse_raw)

# 3) compare to augmented evaluation
print("Reported augmented model raw test MSE on original nodes:", aug_test_mse_on_original)
print("Baseline original nodes MSE:", baseline_orig_mse)
print("Original model MSE from earlier:", globals().get("test_mse", np.nan))

Test sample dates align exactly.
Original model raw test MSE: 114.02058410644531
Reported augmented model raw test MSE on original nodes: 10.83161112482069
Baseline original nodes MSE: 87.27211346028609
Original model MSE from earlier: 114.020584


In [26]:
#do IDP to check a geostatistical model. 
import numpy as np

# Build coordinates for original existing sensors
station_coords = {}
sample_graph = next(iter(graphs.values()))
for node in common_nodes:
    if node in sample_graph.nodes:
        attrs = sample_graph.nodes[node]
        station_coords[node] = (
            float(attrs["station_lat"]),
            float(attrs["station_lon"]),
        )

def idw_predict_day(date_obj, power=2.0, eps=1e-6):
    day_df = grouped[grouped["date"] == date_obj].set_index("location_id")
    preds = {}
    for node in common_nodes:
        if node not in day_df.index or node not in station_coords:
            continue

        lat_i, lon_i = station_coords[node]
        weighted_sum = 0.0
        weight_total = 0.0

        for other in common_nodes:
            if other == node or other not in day_df.index or other not in station_coords:
                continue

            lat_j, lon_j = station_coords[other]
            dist = haversine_km(lat_i, lon_i, lat_j, lon_j)
            if dist < eps:
                continue

            value_j = float(day_df.loc[other, "value"])
            w = 1.0 / (dist**power)
            weighted_sum += w * value_j
            weight_total += w

        if weight_total > 0:
            preds[node] = weighted_sum / weight_total
        else:
            preds[node] = np.nan

    return preds

y_true = []
y_pred = []

for d in test_dates:
    pred_day = idw_predict_day(d, power=2.0)
    day_df = grouped[grouped["date"] == d].set_index("location_id")
    for node in common_nodes:
        if node in day_df.index and node in pred_day:
            pred_val = pred_day[node]
            if np.isnan(pred_val):
                continue
            y_true.append(float(day_df.loc[node, "value"]))
            y_pred.append(pred_val)

y_true = np.array(y_true, dtype=float)
y_pred = np.array(y_pred, dtype=float)

idw_mse = float(np.mean((y_pred - y_true) ** 2))
print(f"IDP / IDW test MSE on existing sensors: {idw_mse:.6f}")
print(f"Number of predictions evaluated: {len(y_true)}")

IDP / IDW test MSE on existing sensors: 24.645659
Number of predictions evaluated: 338


In [ ]:
#IGNORE EVERYTHING AFTER THIS

In [23]:
#the baseline model and getting MSE 
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Dataset
from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv
import numpy as np

# --- 1. ROBUST DATASET DEFINITION ---
class DynamicSpatioTemporalDataset(Dataset):
    def __init__(self, nx_graphs_dict, sorted_dates, pm25_series_dict, window_size=7):
        super().__init__()
        # Fix: Intersect keys to ensure dates exist in both graphs and PM2.5 data
        valid_keys = set(nx_graphs_dict.keys()) & set(pm25_series_dict.keys())
        self.dates = [d for d in sorted_dates if d in valid_keys]
        
        self.graphs_dict = nx_graphs_dict
        self.pm25_dict = pm25_series_dict
        self.window_size = window_size
        
        if len(self.dates) < window_size + 1:
            raise ValueError("Not enough overlapping data between graphs and PM2.5.")
            
        self.node_list = sorted(list(nx_graphs_dict[self.dates[0]].nodes()))

    def len(self):
        return len(self.dates) - self.window_size

    def get(self, idx):
        target_date = self.dates[idx + self.window_size]
        window_dates = self.dates[idx : idx + self.window_size]
        
        sequence_features, edge_indices, edge_weights = [], [], []
        
        for d in window_dates:
            g = self.graphs_dict[d]
            day_feats = [g.nodes[node]['features'] for node in self.node_list]
            sequence_features.append(day_feats)
            
            pyg = from_networkx(g)
            edge_indices.append(pyg.edge_index)
            edge_weights.append(pyg.weight.float())
            
        X = torch.tensor(sequence_features, dtype=torch.float).permute(1, 0, 2)
        raw_y = np.array([self.pm25_dict[target_date].get(n, np.nan) for n in self.node_list])
        pm25_baseline = np.nanmean(raw_y)
        
        y = torch.tensor(raw_y - pm25_baseline, dtype=torch.float).unsqueeze(-1)
        target_g = self.graphs_dict[target_date]
        target_pyg = from_networkx(target_g)
        
        return {
            'x': X, 
            'edge_indices': edge_indices, 
            'edge_weights': edge_weights,
            'y': y,
            'target_edge_index': target_pyg.edge_index,
            'target_edge_weight': target_pyg.weight.float()
        }

# --- 2. DYNAMIC MODEL DEFINITION ---
class DynamicBaselineGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=64):
        super(DynamicBaselineGNN, self).__init__()
        self.gcn_input = GCNConv(num_features, hidden_dim)
        self.gcn_refine = GCNConv(hidden_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_indices, edge_weights, target_edge_index, target_edge_weight):
        num_time_steps = x.shape[1]
        spatial_embeddings = []
        
        # Dynamically loop based on input sequence length
        for t in range(num_time_steps):
            out_t = self.gcn_input(x[:, t, :], edge_indices[t], edge_weights[t])
            spatial_embeddings.append(out_t)
            
        gcn_out = torch.stack(spatial_embeddings, dim=1)
        _, gru_out = self.gru(gcn_out)
        final_embedding = gru_out.squeeze(0)
        final_embedding = self.gcn_refine(final_embedding, target_edge_index, target_edge_weight)
        
        return self.fc(final_embedding)

# --- 3. TRAINING AND EXECUTION ---
def masked_mse_loss(preds, targets):
    mask = ~torch.isnan(targets)
    return nn.functional.mse_loss(preds[mask], targets[mask])

# Config
WINDOW_SIZE = 3
dataset = DynamicSpatioTemporalDataset(graphs, dates, pm25_data, window_size=WINDOW_SIZE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DynamicBaselineGNN(num_features=7, hidden_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Split indices
TOTAL_SAMPLES = len(dataset)
split_point = int(TOTAL_SAMPLES * 0.8)
train_indices = list(range(0, split_point))
test_indices = list(range(split_point, TOTAL_SAMPLES))

# Training Loop
model.train()
for epoch in range(50):
    total_loss = 0
    for i in train_indices:
        batch = dataset[i]
        optimizer.zero_grad()
        
        x = batch['x'].to(device)
        edge_indices = [e.to(device) for e in batch['edge_indices']]
        edge_weights = [w.to(device) for w in batch['edge_weights']]
        y = batch['y'].to(device)
        target_idx = batch['target_edge_index'].to(device)
        target_w = batch['target_edge_weight'].to(device)
        
        out = model(x, edge_indices, edge_weights, target_idx, target_w)
        loss = masked_mse_loss(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Avg Train Loss: {total_loss/len(train_indices):.4f}")

# Evaluation Loop
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for i in test_indices:
        batch = dataset[i]
        out = model(batch['x'].to(device), 
                    [e.to(device) for e in batch['edge_indices']], 
                    [w.to(device) for w in batch['edge_weights']], 
                    batch['target_edge_index'].to(device), 
                    batch['target_edge_weight'].to(device))
        all_preds.append(out.cpu())
        all_targets.append(batch['y'].cpu())

test_mse = torch.mean((torch.cat(all_preds)[~torch.isnan(torch.cat(all_targets))] - 
                       torch.cat(all_targets)[~torch.isnan(torch.cat(all_targets))])**2)
print(f"--- RESULTS ---\nTest Set MSE: {test_mse.item():.4f}")

Epoch 10 | Avg Train Loss: 9.7498
Epoch 20 | Avg Train Loss: 9.6886
Epoch 30 | Avg Train Loss: 10.1489
Epoch 40 | Avg Train Loss: 9.7533
Epoch 50 | Avg Train Loss: 9.7790
--- RESULTS ---
Test Set MSE: 9.8605


In [17]:
def build_augmented_graph(g_orig, cid, cand_data, date_str, grouped_df, dist_threshold=5.0, score_threshold=0.0):
    g_new = g_orig.copy()
    
    # 1. Validate Candidate Data
    c_lat = cand_data.get('lat')
    c_lon = cand_data.get('lon')
    if c_lat is None or c_lon is None:
        return g_new # Skip if candidate has no coords
    c_lat, c_lon = float(c_lat), float(c_lon)
    
    # 2. Add Virtual Node
    sample_node = next(iter(g_orig.nodes(data=True)))[1]
    g_new.add_node(cid, 
                   station_lat=c_lat, 
                   station_lon=c_lon, 
                   features=cand_data['features'],
                   feature_names=sample_node.get('feature_names', []))
    
    # 3. Get Weather for this date
    day_data = grouped_df[grouped_df['date'].astype(str) == date_str]
    # Use fillna to avoid None values in averages
    avg_wind_speed = float(day_data['wind_speed_10m'].mean()) if not day_data.empty else 0.0
    avg_wind_dir = float(day_data['wind_direction_10m'].mean()) if not day_data.empty else 0.0
    
    # 4. Build Edges safely
    for node, node_attr in g_orig.nodes(data=True):
        n_lat = node_attr.get('station_lat')
        n_lon = node_attr.get('station_lon')
        
        # SKIP if node has bad coordinates
        if n_lat is None or n_lon is None:
            continue
            
        dist = haversine_km(c_lat, c_lon, float(n_lat), float(n_lon))
        
        # Check if dist is valid (not None) before comparison
        if dist is not None and dist <= dist_threshold:
            bearing = bearing_deg(c_lat, c_lon, float(n_lat), float(n_lon))
            diff = angle_diff_deg(avg_wind_dir, bearing)
            score = max(math.cos(math.radians(diff)), 0.0) * avg_wind_speed
            
            if score > score_threshold:
                g_new.add_edge(cid, node, weight=score, distance_km=dist)
                g_new.add_edge(node, cid, weight=score, distance_km=dist)
                
    return g_new

In [20]:
import torch
import numpy as np
import networkx as nx
import math
from torch_geometric.utils import from_networkx
import copy

# --- 1. ROBUST MATH HELPERS ---
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    y = math.sin(lon2 - lon1) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def angle_diff_deg(angle1, angle2):
    diff = abs(angle1 - angle2) % 360
    return min(diff, 360 - diff)

# --- 2. AUGMENTED GRAPH BUILDER ---
def build_augmented_graph(g_orig, cid, cand_data, date_str, grouped_df, dist_threshold=5.0, score_threshold=0.0):
    g_new = g_orig.copy()
    c_lat, c_lon = float(cand_data['lat']), float(cand_data['lon'])
    
    # Safely get node attributes
    sample_node = next(iter(g_orig.nodes(data=True)))[1]
    g_new.add_node(cid, 
                   station_lat=c_lat, 
                   station_lon=c_lon, 
                   features=cand_data['features'],
                   feature_names=sample_node.get('feature_names', []))
    
    # Get Date-specific averages
    day_data = grouped_df[grouped_df['date'].astype(str) == date_str]
    avg_wind_speed = float(day_data['wind_speed_10m'].mean()) if not day_data.empty else 0.0
    avg_wind_dir = float(day_data['wind_direction_10m'].mean()) if not day_data.empty else 0.0
    
    # Build Edges
    for node, node_attr in g_orig.nodes(data=True):
        n_lat, n_lon = node_attr.get('station_lat'), node_attr.get('station_lon')
        if n_lat is None or n_lon is None: continue
            
        dist = haversine_km(c_lat, c_lon, float(n_lat), float(n_lon))
        if dist is not None and dist <= dist_threshold:
            bearing = bearing_deg(c_lat, c_lon, float(n_lat), float(n_lon))
            diff = angle_diff_deg(avg_wind_dir, bearing)
            score = max(math.cos(math.radians(diff)), 0.0) * avg_wind_speed
            
            if score > score_threshold:
                g_new.add_edge(cid, node, weight=score, distance_km=dist)
                g_new.add_edge(node, cid, weight=score, distance_km=dist)
    return g_new

# --- 3. MAIN EVALUATION FUNCTION ---
def compute_candidate_improvement_scores(model, base_graphs, processed_grid_by_day, dataset, test_indices, device, grouped):
    model.eval()
    num_orig_nodes = len(dataset.node_list)
    
    # 1. Baseline Run
    print("Computing Baseline MSE...")
    total_baseline_mse = 0
    baseline_count = 0
    with torch.no_grad():
        for i in test_indices:
            batch = dataset[i]
            out = model(batch['x'].to(device), 
                        [e.to(device) for e in batch['edge_indices']], 
                        [w.to(device) for w in batch['edge_weights']], 
                        batch['target_edge_index'].to(device), 
                        batch['target_edge_weight'].to(device))
            y = batch['y'].to(device)
            mask = ~torch.isnan(y)
            total_baseline_mse += torch.sum((out[mask] - y[mask])**2).item()
            baseline_count += mask.sum().item()
    baseline_mse = total_baseline_mse / baseline_count
    print(f"Baseline MSE: {baseline_mse:.6f}")

    # 2. Evaluation Loop
    improvement_scores = {}
    all_candidates = list(processed_grid_by_day[list(processed_grid_by_day.keys())[0]].keys())
    
    for cid in all_candidates:
        candidate_mse_list = []
        
        for i in test_indices:
            date = dataset.dates[i + dataset.window_size]
            date_str = str(date)
            cand_data = processed_grid_by_day[date_str].get(cid)
            if cand_data is None: continue
            
            # Augment Topology
            g_aug = build_augmented_graph(base_graphs[date], cid, cand_data, date_str, grouped)
            pyg_aug = from_networkx(g_aug)
            
            # Manually Inject Features (Fixing Shape Mismatch)
            batch = dataset[i]
            x_base = batch['x'].to(device)
            cand_x = torch.tensor(cand_data['features'], dtype=torch.float).to(device)
            cand_x_expanded = cand_x.view(1, 1, -1).expand(-1, x_base.shape[1], -1)
            x_aug = torch.cat([x_base, cand_x_expanded], dim=0)
            
            # Inference
            aug_edge_idx = pyg_aug.edge_index.to(device)
            aug_edge_w = pyg_aug.weight.float().to(device)
            
            with torch.no_grad():
                out = model(x_aug, 
                            [aug_edge_idx] * batch['x'].shape[1], 
                            [aug_edge_w] * batch['x'].shape[1], 
                            aug_edge_idx, 
                            aug_edge_w)
                
                # Mask to Original Sensors
                real_out = out[:num_orig_nodes]
                y = batch['y'].to(device)
                mask = ~torch.isnan(y)
                
                if mask.any():
                    mse = torch.sum((real_out[mask] - y[mask])**2).item()
                    candidate_mse_list.append(mse / mask.sum().item())
        
        if candidate_mse_list:
            avg_aug_mse = np.mean(candidate_mse_list)
            improvement_scores[cid] = baseline_mse - avg_aug_mse
            
    # 3. Output
    sorted_results = sorted(improvement_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Execute:
results = compute_candidate_improvement_scores(model, graphs, processed_grid_by_day, dataset, test_indices, device, grouped)

Computing Baseline MSE...
Baseline MSE: 9.081419


In [22]:
results

[(136, np.float64(-0.21809384005886834)),
 (110, np.float64(-0.2290290565757509)),
 (67, np.float64(-0.27620244859815557)),
 (124, np.float64(-0.32859273790479726)),
 (18, np.float64(-0.3568275905155609)),
 (72, np.float64(-0.35776481228274726)),
 (123, np.float64(-0.3587453962206002)),
 (29, np.float64(-0.3700775519951254)),
 (109, np.float64(-0.370084509149299)),
 (17, np.float64(-0.3733473851130551)),
 (33, np.float64(-0.37725402992088597)),
 (45, np.float64(-0.37747183312903054)),
 (32, np.float64(-0.379260509997815)),
 (56, np.float64(-0.3809146587665264)),
 (96, np.float64(-0.38302743818376506)),
 (89, np.float64(-0.38873710632324254)),
 (55, np.float64(-0.39451618327961135)),
 (46, np.float64(-0.3949232354864378)),
 (95, np.float64(-0.3963810100422034)),
 (66, np.float64(-0.39953859769380884)),
 (59, np.float64(-0.39988162967708796)),
 (31, np.float64(-0.40321815063903443)),
 (87, np.float64(-0.40388002729082473)),
 (111, np.float64(-0.40516156816816107)),
 (85, np.float64(-0.40

In [ ]:
# 1. Define baseline_mse (Must be the same value used in compute_candidate_improvement_scores)
# Assuming baseline_mse is already in your scope from the previous step

# 2. Process and Filter
# results is a list of tuples: [(cid, gain_val), ...]
# where gain_val = (baseline_mse - augmented_mse)

cleaned_results = []
for cid, raw_gain in results:
    # Only keep results where the gain was positive (MSE actually went down)
    if raw_gain > 0:
        improvement_pct = (raw_gain / baseline_mse) * 100
        cleaned_results.append((cid, improvement_pct))

# 3. Sort by percentage improvement descending
cleaned_results.sort(key=lambda x: x[1], reverse=True)

# 4. View Top 10
print("--- Top 10 High-Value Sensor Locations ---")
for cid, pct in cleaned_results[:10]:
    print(f"ID: {cid} | Improvement: {pct:.2f}%")

In [ ]:
import folium
import branca.colormap as cm
import numpy as np
from folium.features import DivIcon

# --- 1. ROBUST WIND AVERAGING ---
def get_aligned_mean_wind(cid, processed_grid_by_day, test_indices, dates, w_size):
    """
    Calculates average wind direction by extracting it from the nested feature array.
    """
    test_dates = [dates[i + w_size] for i in test_indices]
    sin_sum, cos_sum = 0, 0
    count = 0
    
    for date in test_dates:
        date_str = str(date)
        if date_str in processed_grid_by_day and cid in processed_grid_by_day[date_str]:
            entry = processed_grid_by_day[date_str][cid]
            
            # Access the nested structure
            names = entry.get('feature_names', [])
            feats = entry.get('features', [])
            
            # Find the index of 'wind_direction_10m'
            if 'wind_direction_10m' in names:
                idx = names.index('wind_direction_10m')
                deg = feats[idx]
                
                # Perform the math
                rad = np.radians(float(deg))
                sin_sum += np.sin(rad)
                cos_sum += np.cos(rad)
                count += 1
            
    if count == 0: return None
    # Calculate vector mean and convert back to degrees
    return np.degrees(np.arctan2(sin_sum/count, cos_sum/count)) % 360

# --- 2. MAP INITIALIZATION ---
WINDOW_SIZE = 3
scores = [val for cid, val in results]
colormap = cm.LinearColormap(colors=['red', 'white', 'green'], vmin=min(scores), vmax=max(scores))
colormap.caption = 'Network Improvement Score (MSE Reduction)'

m = folium.Map(location=[36.5, 127.5], zoom_start=7, tiles="CartoDB positron")

# --- 3. PLOTTING LOOP ---
for cid, score in results:
    if cid in coords_map:
        lat, lon = coords_map[cid]
        
        # Plot Gain Circle
        folium.CircleMarker(
            location=[lat, lon],
            radius=8,
            color=colormap(score),
            fill=True,
            fill_color=colormap(score),
            fill_opacity=0.7,
            popup=f"ID: {cid}<br>Gain: {score:.4f}",
            tooltip=f"ID: {cid}"
        ).add_to(m)
        
        # Plot Wind Arrow (with safety check)
        wind_dir = get_aligned_mean_wind(cid, processed_grid_by_day, test_indices, dates, WINDOW_SIZE)
        if wind_dir is not None:
            rotation = wind_dir + 180 
            icon_html = f'<div style="transform: rotate({rotation}deg); font-size: 12px; color: #333;">&#10148;</div>'
            
            folium.Marker(
                location=[lat, lon],
                icon=DivIcon(
                    icon_size=(20,20),
                    icon_anchor=(10,10),
                    html=icon_html
                ),
                popup=f"ID: {cid}<br>Avg Wind Dir: {wind_dir:.1f}°"
            ).add_to(m)

# --- 4. FINALIZE ---
m.add_child(colormap)
m.save("sensor_optimization_map.html")
print("Map successfully saved as 'sensor_optimization_map.html'")

In [ ]:
# Pick one cid from your results to test
sample_cid = results[0][0] 

print(f"Testing diagnostics for CID: {sample_cid}")
test_dates = [dates[i + WINDOW_SIZE] for i in test_indices]

found_data = False
for date in test_dates[:5]: # Check the first 5 test dates
    date_str = str(date)
    if date_str in processed_grid_by_day:
        if sample_cid in processed_grid_by_day[date_str]:
            data = processed_grid_by_day[date_str][sample_cid]
            val = data.get('wind_direction_10m')
            print(f"Date: {date_str} | Data found: {val}")
            if val is not None:
                found_data = True
        else:
            print(f"Date: {date_str} | CID {sample_cid} not in grid.")
    else:
        print(f"Date: {date_str} | Date not in processed_grid_by_day.")

if not found_data:
    print("CRITICAL: No wind data found for this CID in the test dates!")

In [ ]:
processed_grid_by_day['2025-07-08'][116]

In [ ]:
import folium
import branca.colormap as cm

# 1. Setup Data for Plotting
# Assuming 'results' is your list of tuples: [(116, 0.262...), (72, 0.262...), ...]
# and 'processed_grid_by_day' has the lat/lon info.
# We extract one date (the first) to get the coordinate lookup table
first_date = list(processed_grid_by_day.keys())[0]
coords_map = {cid: (data['lat'], data['lon']) for cid, data in processed_grid_by_day[first_date].items()}

# 2. Setup Color Map
scores = [val for cid, val in results]
colormap = cm.LinearColormap(colors=['red', 'white', 'green'], vmin=min(scores), vmax=max(scores))
colormap.caption = 'Network Improvement Score (MSE Reduction)'

# 3. Initialize Map (Centered on South Korea)
m = folium.Map(location=[36.5, 127.5], zoom_start=7, tiles="CartoDB positron")

# 4. Add Markers
for cid, score in results:
    if cid in coords_map:
        lat, lon = coords_map[cid]
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            color=colormap(score),
            fill=True,
            fill_color=colormap(score),
            fill_opacity=0.8,
            popup=f"ID: {cid}<br>Gain: {score:.4f}",
            tooltip=f"ID: {cid}"
        ).add_to(m)

# 5. Add Color Legend
m.add_child(colormap)

# Save or display
m.save("sensor_optimization_map.html")
print("Map saved as sensor_optimization_map.html")

In [ ]:
# Diagnostic Check
print(f"Num nodes in G_aug: {g_aug.number_of_nodes()}")
print(f"Shape of X: {x_base.shape}")

In [ ]:
import torch
import networkx as nx
import math
import numpy as np
from torch_geometric.utils import from_networkx

# --- 1. SPATIAL & GRAPH LOGIC (Reconfirmed) ---
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    y = math.sin(lon2 - lon1) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def angle_diff_deg(angle1, angle2):
    diff = abs(angle1 - angle2) % 360
    return min(diff, 360 - diff)

# --- 2. HARDENED AUGMENTED GRAPH BUILDER ---
def build_augmented_graph(g_orig, cid, cand_data, date_str, grouped_df, dist_threshold=5.0, score_threshold=0.0):
    g_new = g_orig.copy()
    
    # DYNAMIC SCHEMA SYNC: Extract existing keys from a representative node to ensure structure matches
    # This fixes the ValueError: "Not all nodes contain the same attributes"
    sample_node_key, sample_node_attrs = next(iter(g_orig.nodes(data=True)))
    
    # Build attributes dict for the virtual node
    virtual_node_attrs = {
        'station_lat': float(cand_data['lat']),
        'station_lon': float(cand_data['lon']),
        'features': cand_data['features']
    }
    # Add other attributes from existing nodes (e.g., 'feature_names')
    for k, v in sample_node_attrs.items():
        if k not in virtual_node_attrs:
            virtual_node_attrs[k] = v
            
    g_new.add_node(cid, **virtual_node_attrs)
    
    # Calculate daily weather metrics for the edge weights (Proxy for virtual node)
    day_data = grouped_df[grouped_df['date'].astype(str) == date_str]
    # We use the daily average as the wind behavior for the virtual node location
    avg_wind_speed = day_data['wind_speed_10m'].mean()
    avg_wind_dir = day_data['wind_direction_10m'].mean()
    
    # Build Edges (Virtual <-> Existing)
    for node, node_attr in g_orig.nodes(data=True):
        lat1, lon1 = virtual_node_attrs['station_lat'], virtual_node_attrs['station_lon']
        lat2, lon2 = node_attr['station_lat'], node_attr['station_lon']
        
        dist = haversine_km(lat1, lon1, lat2, lon2)
        
        if dist <= dist_threshold:
            bearing = bearing_deg(lat1, lon1, lat2, lon2)
            diff = angle_diff_deg(avg_wind_dir, bearing)
            
            # Logic strictly mirroring your provided training construction
            score = max(math.cos(math.radians(diff)), 0.0) * avg_wind_speed
            
            if score > score_threshold:
                g_new.add_edge(cid, node, weight=score, distance_km=dist)
                g_new.add_edge(node, cid, weight=score, distance_km=dist)
                
    return g_new

# --- 3. EVALUATION PIPELINE ---
candidate_mse_tracker = {}
SCORE_THRESHOLD = 0.0 # Define this as per your original training

model.eval()
print(f"Evaluation started for {len(processed_grid_by_day)} candidate entries...")

with torch.no_grad():
    for i in test_indices:
        date = dates[i + WINDOW_SIZE]
        date_str = str(date)
        
        # Guard: Only evaluate if data exists
        if date_str not in processed_grid_by_day or date not in graphs:
            continue
            
        g_orig = graphs[date]
        base_batch = dataset[i]
        
        # Process every candidate for this date
        for cid, cand_data in processed_grid_by_day[date_str].items():
            
            # 1. Build Graph with identical logic
            g_aug = build_augmented_graph(g_orig, cid, cand_data, date_str, grouped, score_threshold=SCORE_THRESHOLD)
            pyg = from_networkx(g_aug)
            
            # 2. Input Preparation
            x_base = base_batch['x'].to(device)
            # Create virtual features tensor
            v_feats = torch.tensor(cand_data['features']).float().to(device).view(1, 1, -1)
            v_feats_expanded = v_feats.expand(-1, x_base.shape[1], -1)
            # Concat virtual features: [N_original, Time, F] -> [N+1, Time, F]
            x_aug = torch.cat([x_base, v_feats_expanded], dim=0) 
            
            # 3. Model Inference (Pre-trained)
            edge_idx = pyg.edge_index.to(device)
            edge_w = pyg.weight.float().to(device)
            
            # Using the exact same model forward call
            out = model(x_aug, [edge_idx]*WINDOW_SIZE, [edge_w]*WINDOW_SIZE, edge_idx, edge_w)
            
            # 4. Metrics (Exclude Virtual Node)
            y = base_batch['y'].to(device)
            preds = out[:-1] 
            mask = ~torch.isnan(y)
            
            if mask.any():
                mse = torch.mean((preds[mask] - y[mask])**2).item()
                if cid not in candidate_mse_tracker:
                    candidate_mse_tracker[cid] = []
                candidate_mse_tracker[cid].append(mse)

# --- 4. TOP 5 REPORT ---
print("\n" + "="*40)
print("TOP 5 CANDIDATES (Avg Test MSE)")
print("="*40)

# Filter out candidates with no valid data
final_results = {cid: np.mean(mses) for cid, mses in candidate_mse_tracker.items() if len(mses) > 0}
sorted_candidates = sorted(final_results.items(), key=lambda x: x[1])

for rank, (cid, mse) in enumerate(sorted_candidates[:5], 1):
    print(f"Rank {rank}: ID {cid} | Avg Test MSE: {mse:.4f}")

In [ ]:
import numpy as np
import pandas as pd

# 1. Pre-process the dataframe to create a searchable string date column
# Do this once before the loop to save time
grouped['date_str'] = grouped['date'].apply(lambda x: x.isoformat())

# 2. Re-define the IDW prediction function
def idw_predict(target_lat, target_lon, neighbor_data, p=2):
    lat1, lon1 = target_lat, target_lon
    lat2, lon2 = neighbor_data['station_lat'].values, neighbor_data['station_lon'].values
    dists = np.array([haversine_km(lat1, lon1, lt, ln) for lt, ln in zip(lat2, lon2)])
    
    # Avoid division by zero
    dists = np.where(dists < 1e-6, 1e-6, dists)
    
    weights = 1.0 / (dists ** p)
    weights /= weights.sum()
    
    # Weighted average of features
    neighbor_vals = np.stack(neighbor_data['final_features'].values)
    prediction = np.dot(weights, neighbor_vals)
    return prediction

# 3. IDW Evaluation Pipeline
idw_mse_scores = []
predictor_cols = [
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'wind_speed_10m','wind_direction_10m','surface_pressure','precipitation'
]

print(f"Starting IDW baseline on {len(test_indices)} test dates...")

for i in test_indices:
    target_date_str = dates[i + WINDOW_SIZE]
    
    # Now this match will work perfectly
    day = grouped[grouped['date_str'] == target_date_str].reset_index(drop=True)
    
    if day.empty:
        continue
    
    # Data cleaning (same as your graph logic)
    raw_day_feats = day[predictor_cols].values
    regional_baseline = np.nanmean(raw_day_feats, axis=0)
    
    final_features_list = []
    for _, row in day.iterrows():
        raw_feat = np.array([float(row[col]) if col in row and not pd.isna(row[col]) else np.nan for col in predictor_cols])
        filled_feat = np.where(np.isnan(raw_feat), regional_baseline, raw_feat)
        final_features_list.append(filled_feat)
    
    day['final_features'] = final_features_list
    
    # IDW Calculation
    for idx, row in day.iterrows():
        # Get neighbors (everyone else)
        neighbors = day.drop(idx)
        if len(neighbors) == 0: continue
        
        pred = idw_predict(row['station_lat'], row['station_lon'], neighbors)
        
        # Calculate Squared Error
        sq_err = (pred - row['final_features']) ** 2
        idw_mse_scores.append(np.nanmean(sq_err))

# 4. Final Report
if idw_mse_scores:
    avg_idw_mse = np.nanmean(idw_mse_scores)
    print("\n" + "="*40)
    print(f"GEOSPATIAL BASELINE (IDW)")
    print(f"Mean Test Set MSE: {avg_idw_mse:.4f}")
    print("="*40)
else:
    print("\nError: Still no matches. Ensure 'dates' list contains strings in 'YYYY-MM-DD' format.")

In [ ]:
import copy
import torch
import torch.optim as optim
import math
import numpy as np

# --- 1. AUGMENTATION & SCHEMA SYNC ---
def augment_and_standardize(graphs, processed_grid_by_day, best_cid, grouped_df):
    aug_graphs = copy.deepcopy(graphs)
    
    def standardize(G):
        all_keys = set()
        for _, attrs in G.nodes(data=True): all_keys.update(attrs.keys())
        for _, attrs in G.nodes(data=True):
            for key in all_keys:
                if key not in attrs: attrs[key] = 0.0
        return G

    print(f"Injecting Node {best_cid}...")
    for date_str, g in aug_graphs.items():
        if date_str in processed_grid_by_day and best_cid in processed_grid_by_day[date_str]:
            cand = processed_grid_by_day[date_str][best_cid]
            g.add_node(best_cid, station_lat=float(cand['lat']), 
                       station_lon=float(cand['lon']), features=np.array(cand['features']))
            
            day_data = grouped_df[grouped_df['date'].astype(str) == date_str]
            avg_wind_speed = day_data['wind_speed_10m'].mean() if not day_data.empty else 1.0
            avg_wind_dir = day_data['wind_direction_10m'].mean() if not day_data.empty else 0.0
            
            for node, n_attr in list(g.nodes(data=True)):
                if node == best_cid: continue
                dist = haversine_km(cand['lat'], cand['lon'], n_attr['station_lat'], n_attr['station_lon'])
                if dist <= 5.0: 
                    bearing = bearing_deg(cand['lat'], cand['lon'], n_attr['station_lat'], n_attr['station_lon'])
                    diff = angle_diff_deg(avg_wind_dir, bearing)
                    score = max(math.cos(math.radians(diff)), 0.0) * avg_wind_speed
                    if score > 0.0:
                        g.add_edge(best_cid, node, weight=score, distance_km=dist)
                        g.add_edge(node, best_cid, weight=score, distance_km=dist)
        
        aug_graphs[date_str] = standardize(g)
    return aug_graphs

# --- 2. PREP DATASET ---
best_cid = sorted_candidates[0][0] 
aug_graphs = augment_and_standardize(graphs, processed_grid_by_day, best_cid, grouped)
aug_dataset = DynamicSpatioTemporalDataset(aug_graphs, dates, pm25_data, window_size=WINDOW_SIZE)

# --- 3. FINE-TUNING CONFIGURATION ---
model.train()

# Freeze the GRU layers (keep temporal memory intact)
for param in model.gru.parameters():
    param.requires_grad = False

# Re-initialize optimizer for Spatial layers only (GCNs)
# Learning rate 1e-4 is gentle for fine-tuning
fine_tune_optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

print("Starting Fine-Tuning...")

# --- 4. FINE-TUNING LOOP ---
for epoch in range(50):
    total_loss = 0
    for i in train_indices:
        batch = aug_dataset[i]
        fine_tune_optimizer.zero_grad()
        
        # Forward pass
        out = model(batch['x'].to(device), 
                    [e.to(device) for e in batch['edge_indices']], 
                    [w.to(device) for w in batch['edge_weights']], 
                    batch['target_edge_index'].to(device), 
                    batch['target_edge_weight'].to(device))
        
        loss = masked_mse_loss(out, batch['y'].to(device))
        loss.backward()
        fine_tune_optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1} | Fine-Tune Loss: {total_loss/len(train_indices):.4f}")

# --- 5. EVALUATION ---
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for i in test_indices:
        batch = aug_dataset[i]
        out = model(batch['x'].to(device), 
                    [e.to(device) for e in batch['edge_indices']], 
                    [w.to(device) for w in batch['edge_weights']], 
                    batch['target_edge_index'].to(device), 
                    batch['target_edge_weight'].to(device))
        all_preds.append(out.cpu())
        all_targets.append(batch['y'].cpu())

# Calculate MSE
all_preds_cat = torch.cat(all_preds)
all_targets_cat = torch.cat(all_targets)
mask = ~torch.isnan(all_targets_cat)
test_mse = torch.mean((all_preds_cat[mask] - all_targets_cat[mask])**2)

print(f"\n--- FINAL RESULTS ---")
print(f"Fine-Tuned Test Set MSE: {test_mse.item():.4f}")

In [ ]:
import random

def evaluate_candidate(cid, graphs, processed_grid_by_day, grouped_df, model, device, dates, pm25_data, window_size, test_indices):
    """Wraps the injection and inference process into a single scoring function."""
    
    # 1. Augment with specific candidate
    aug_graphs = augment_and_standardize(graphs, processed_grid_by_day, cid, grouped_df)
    
    # 2. Build Dataset
    dataset = DynamicSpatioTemporalDataset(aug_graphs, dates, pm25_data, window_size=window_size)
    
    # 3. Inference Only
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for i in test_indices:
            batch = dataset[i]
            out = model(batch['x'].to(device), 
                        [e.to(device) for e in batch['edge_indices']], 
                        [w.to(device) for w in batch['edge_weights']], 
                        batch['target_edge_index'].to(device), 
                        batch['target_edge_weight'].to(device))
            all_preds.append(out.cpu())
            all_targets.append(batch['y'].cpu())
            
    # 4. Calculate MSE
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    mask = ~torch.isnan(targets)
    mse = torch.mean((preds[mask] - targets[mask])**2).item()
    return mse

# --- Execution ---

# 1. Identify IDs
best_cid = sorted_candidates[0][0]
worst_cid = sorted_candidates[-1][0]
random_cid = random.choice([c[0] for c in sorted_candidates])

candidates_to_test = {
    "Best Candidate": best_cid,
    "Random Candidate": random_cid,
    "Worst Candidate": worst_cid
}

results = {}

# 2. Run the Comparison
print(f"Starting comparative validation (Inference-only)...")
for label, cid in candidates_to_test.items():
    mse = evaluate_candidate(cid, graphs, processed_grid_by_day, grouped, model, device, dates, pm25_data, WINDOW_SIZE, test_indices)
    results[label] = mse
    print(f"{label} (ID: {cid}) | Test MSE: {mse:.4f}")

# 3. Final Summary Table
print("\n" + "="*30)
print(f"{'Location Strategy':<20} | {'MSE'}")
print("-"*30)
for label, mse in results.items():
    print(f"{label:<20} | {mse:.4f}")
print("="*30)

In [ ]:
import torch
import copy
import numpy as np
import networkx as nx
import math

# --- HELPER: YOUR ORIGINAL MATH ---
def haversine_km(lat1, lon1, lat2, lon2):
    # Ensure you have this function defined (or import from your project)
    ... 

def bearing_deg(lat1, lon1, lat2, lon2):
    # Ensure you have this function defined
    ...

def angle_diff_deg(angle1, angle2):
    # Ensure you have this function defined
    ...

def evaluate_all_candidate_locations(model, base_graphs, processed_grid_by_day, dataset, test_indices, device, SCORE_THRESHOLD=0.5):
    """
    Injects a virtual candidate and evaluates network performance using 
    the original wind/distance edge topology rules.
    """
    model.eval()
    all_candidates = list(processed_grid_by_day[list(processed_grid_by_day.keys())[0]].keys())
    results = {}
    
    DIST_THRESHOLD_KM = 5.0

    print(f"Evaluating {len(all_candidates)} candidates...")
    
    for cid in all_candidates:
        aug_graphs = {}
        for date, G in base_graphs.items():
            G_aug = copy.deepcopy(G)
            grid_info = processed_grid_by_day[date].get(cid)
            if grid_info is None: continue
            
            # 1. Add Candidate Node
            G_aug.add_node(cid, 
                           station_lat=float(grid_info['lat']), 
                           station_lon=float(grid_info['lon']))
            
            # 2. Connect Candidate (Bidirectional Logic: Candidate <-> Existing)
            cand_lat, cand_lon = float(grid_info['lat']), float(grid_info['lon'])
            cand_wind_spd = float(grid_info['wind_speed_10m'])
            cand_wind_dir = float(grid_info['wind_direction_10m'])
            
            for sensor_id in G.nodes():
                s_node = G.nodes[sensor_id]
                s_lat, s_lon = float(s_node['station_lat']), float(s_node['station_lon'])
                
                # Retrieve sensor wind (assumes you can access this from your feature vector or metadata)
                # Note: Adjust this lookup to your specific data structure
                s_wind_spd = float(s_node['wind_speed_10m']) 
                s_wind_dir = float(s_node['wind_direction_10m'])
                
                dist = haversine_km(cand_lat, cand_lon, s_lat, s_lon)
                
                if dist <= DIST_THRESHOLD_KM:
                    # --- Edge: Candidate -> Sensor ---
                    bearing_c2s = bearing_deg(cand_lat, cand_lon, s_lat, s_lon)
                    diff_c2s = angle_diff_deg(cand_wind_dir, bearing_c2s)
                    score_c2s = max(math.cos(math.radians(diff_c2s)), 0.0) * cand_wind_spd
                    
                    if score_c2s > SCORE_THRESHOLD:
                        G_aug.add_edge(cid, sensor_id, weight=score_c2s, distance_km=dist)
                        
                    # --- Edge: Sensor -> Candidate ---
                    bearing_s2c = bearing_deg(s_lat, s_lon, cand_lat, cand_lon)
                    diff_s2c = angle_diff_deg(s_wind_dir, bearing_s2c)
                    score_s2c = max(math.cos(math.radians(diff_s2c)), 0.0) * s_wind_spd
                    
                    if score_s2c > SCORE_THRESHOLD:
                        G_aug.add_edge(sensor_id, cid, weight=score_s2c, distance_km=dist)
            
            aug_graphs[date] = G_aug

        # 3. Inference
        aug_mse = run_inference_on_graphs(model, aug_graphs, dataset, test_indices, device)
        results[cid] = baseline_mse - aug_mse
        
    return results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

def plot_sensor_network_with_wind(grouped_df, top_candidates, all_candidate_data):
    # 1. Aggregate Wind Data by Location
    # We group by lat/lon to get the average conditions per station
    avg_weather = grouped_df.groupby(['station_lat', 'station_lon']).agg({
        'wind_speed_10m': 'mean',
        'wind_direction_10m': 'mean'
    }).reset_index()

    # Convert Meteorological Degrees to Cartesian U, V (Speed)
    # Wind direction is 'from', we want arrow pointing 'to'
    rads = np.radians(avg_weather['wind_direction_10m'])
    avg_weather['u'] = -avg_weather['wind_speed_10m'] * np.sin(rads)
    avg_weather['v'] = -avg_weather['wind_speed_10m'] * np.cos(rads)

    # 2. Extract Candidates
    top_ids = [c[0] for c in top_candidates[:5]]
    cand_lats, cand_lons = [], []
    for cid in top_ids:
        for date_str in all_candidate_data:
            if cid in all_candidate_data[date_str]:
                cand_lats.append(all_candidate_data[date_str][cid]['lat'])
                cand_lons.append(all_candidate_data[date_str][cid]['lon'])
                break

    # 3. Plotting
    plt.figure(figsize=(12, 9))
    
    # Plot Wind Vectors (Quiver)
    plt.quiver(avg_weather['station_lon'], avg_weather['station_lat'], 
               avg_weather['u'], avg_weather['v'], 
               color='gray', alpha=0.4, label='Avg Wind Flow', scale=50)

    # Plot Existing Sensors
    plt.scatter(avg_weather['station_lon'], avg_weather['station_lat'], 
                color='blue', alpha=0.6, label='Existing Sensors', s=40)
    
    # Plot Candidates
    plt.scatter(cand_lons, cand_lats, 
                color='red', marker='*', s=250, label='Top 5 Virtual Candidates', edgecolors='black')

    # Annotations
    for i, cid in enumerate(top_ids):
        plt.annotate(f" #{i+1}", (cand_lons[i], cand_lats[i]), 
                     fontsize=12, fontweight='bold', xytext=(8, 8), textcoords='offset points')

    plt.title("Sensor Network & Wind Flow Analysis")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# Run it
plot_sensor_network_with_wind(grouped, sorted_candidates, processed_grid_by_day)

In [ ]:
# 1. Create the PM2.5 lookup dictionary
pm25_series_dict = {}

for d in grouped['date'].unique():
    date_key = d.isoformat() if hasattr(d, 'isoformat') else str(d)
    day_df = grouped[grouped['date'] == d]
    
    # Map location_id -> pm2.5 for this specific day
    # (Replace 'pm25_column_name' with your actual column name, e.g., 'pm2.5' or 'pm25')
    pm25_series_dict[date_key] = dict(zip(day_df['location_id'], day_df['value']))

# 2. Instantiate your PyTorch Geometric Dataset
WINDOW_SIZE = 7

dataset = SpatioTemporalGraphDataset(
    nx_graphs_dict=graphs,
    sorted_dates=dates,
    pm25_series_dict=pm25_series_dict,
    window_size=WINDOW_SIZE
)

print(f"Total sequences generated by rolling window: {len(dataset)}")

# 3. Calculate chronological sequence indices
total_sequences = len(dataset)

# Let's do a standard 70% Train / 15% Validation / 15% Test split
train_end = int(total_sequences * 0.70)
val_end = int(total_sequences * 0.85)

train_indices = list(range(0, train_end))
val_indices = list(range(train_end, val_end))
test_indices = list(range(val_end, total_sequences))

print(f"Train samples: {len(train_indices)} days | ({dates[0]} to {dates[train_end + WINDOW_SIZE - 1]})")
print(f"Val samples  : {len(val_indices)} days | ({dates[train_end + WINDOW_SIZE]} to {dates[val_end + WINDOW_SIZE - 1]})")
print(f"Test samples : {len(test_indices)} days | ({dates[val_end + WINDOW_SIZE]} to {dates[-1]})")

In [ ]:
#train the model 
# 4. Train the baseline model
trained_model = train_baseline_model(
    dataset=dataset,
    train_indices=train_indices,
    val_indices=val_indices,
    epochs=30,
    lr=0.001
)

In [ ]:
from torch_geometric.loader import DataLoader

# Create a data loader strictly for the unseen test split
test_set = [dataset[i] for i in test_indices]
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

trained_model.eval()

all_predictions = []
all_ground_truth = []
test_mse_accumulator = 0.0
total_valid_nodes = 0

with torch.no_grad():
    for batch in test_loader:
        # Predict tomorrow's PM2.5 using only historical weather maps
        out = trained_model(batch.x, batch.edge_index, batch.edge_attr)
        
        # Mask out any stations that contain missing/NaN ground truth targets
        mask = ~torch.isnan(batch.y)
        
        if mask.sum() > 0:
            pred_filtered = out[mask]
            true_filtered = batch.y[mask]
            
            all_predictions.extend(pred_filtered.cpu().numpy().flatten())
            all_ground_truth.extend(true_filtered.cpu().numpy().flatten())

# Calculate overall performance metrics
all_predictions = np.array(all_predictions)
all_ground_truth = np.array(all_ground_truth)

final_mse = np.mean((all_predictions - all_ground_truth) ** 2)
final_rmse = np.sqrt(final_mse)
final_mae = np.mean(np.abs(all_predictions - all_ground_truth))

print("\n================ TEST SET PERFORMANCE ================")
print(f"Baseline Network Mean Squared Error (MSE)      : {final_mse:.4f}")
print(f"Baseline Network Root Mean Squared Error (RMSE): {final_rmse:.4f}")
print(f"Baseline Network Mean Absolute Error (MAE)     : {final_mae:.4f}")
print("======================================================")

In [ ]:
import copy
import numpy as np
import torch
import networkx as nx
from torch.utils.data import DataLoader

# --- 1. SETUP: Extract Sensor Data ---
# We extract this once so we don't have to re-scan the graph
first_date = list(graphs.keys())[0]
sample_g = graphs[first_date]
# Build dict of {id: {'station_lat': ..., 'station_lon': ...}}
sensor_coords = {
    node: {'station_lat': attrs['station_lat'], 'station_lon': attrs['station_lon']}
    for node, attrs in sample_g.nodes(data=True)
}

# --- 2. HELPERS ---
def add_virtual_node_to_graph(graph, cell_data, sensor_coords):
    """Injects one virtual node into a single graph instance."""
    g = graph.copy()
    
    # 1. Get Template for Schema (to avoid ValueError)
    template_node = list(g.nodes())[0]
    template_attrs = g.nodes[template_node]
    
    # 2. Create Virtual Node with required attributes
    v_attrs = {
        'station_lat': cell_data['lat'],
        'station_lon': cell_data['lon'],
        'features': cell_data['features'],
        'feature_names': template_attrs['feature_names']
    }
    g.add_node(cell_data['cell_id'], **v_attrs)
    
    # 3. Connect (5km cutoff)
    for existing_id, existing_coords in sensor_coords.items():
        # Calculate distance
        dist = haversine(
            cell_data['lat'], cell_data['lon'],
            existing_coords['station_lat'], existing_coords['station_lon']
        )
        
        if dist <= 5.0:
            # Add edge
            weight = calculate_wind_weight(
                (cell_data['lat'], cell_data['lon']),
                (existing_coords['station_lat'], existing_coords['station_lon']),
                cell_data['features']
            )
            g.add_edge(cell_data['cell_id'], existing_id, weight=weight)
    return g

def evaluate_pipeline(model, original_graphs, candidate_data_by_date):
    """
    Augments graphs, runs eval, returns MSE.
    candidate_data_by_date: dict of {date: grid_entry}
    """
    # Create augmented graph dict
    aug_graphs = {}
    for date, g in original_graphs.items():
        if date in candidate_data_by_date:
            aug_graphs[date] = add_virtual_node_to_graph(g, candidate_data_by_date[date], sensor_coords)
        else:
            aug_graphs[date] = g
            
    # Relabel all to strings to avoid TypeErrors
    for date in aug_graphs:
        aug_graphs[date] = nx.relabel_nodes(aug_graphs[date], str)
    
    # Evaluate
    model.eval()
    all_preds, all_targets = [], []
    
    # Dataset call
    eval_ds = DynamicSpatioTemporalDataset(aug_graphs, test_dates, pm25_data, window_size=3)
    
    with torch.no_grad():
        for batch in DataLoader(eval_ds, batch_size=1):
            # Forward pass
            out = model(batch['x'].to(device), [e.to(device) for e in batch['edge_indices']], 
                        [w.to(device) for w in batch['edge_weights']], 
                        batch['target_edge_index'].to(device), batch['target_edge_weight'].to(device))
            
            y = batch['y'].to(device)
            # Mask out the virtual node (assuming it has no ground truth, it is likely NaN)
            mask = ~torch.isnan(y)
            if mask.sum() > 0:
                all_preds.append(out[mask].cpu())
                all_targets.append(y[mask].cpu())
                
    if not all_preds: return float('inf')
    return torch.mean((torch.cat(all_preds) - torch.cat(all_targets))**2).item()

# --- 3. MAIN SEARCH LOOP ---
baseline_mse = 11.5 # (Ensure this is your actual baseline)
results = {}

# Flatten the grid data into {candidate_id: {date: data}}
all_candidates = {}
for date, grid_items in processed_grid_by_day.items():
    for _, data in grid_items.items():
        cid = data['cell_id']
        if cid not in all_candidates: all_candidates[cid] = {}
        all_candidates[cid][date] = data

print(f"Evaluating {len(all_candidates)} candidates...")

for cid, cand_data in all_candidates.items():
    try:
        mse = evaluate_pipeline(model, graphs, cand_data)
        results[cid] = mse - baseline_mse
        print(f"Candidate {cid}: Change {results[cid]:.4f}")
    except Exception as e:
        print(f"Skipping {cid}: {e}")

# --- 4. TOP 5 ---
top_5 = sorted(results.items(), key=lambda x: x[1])[:5]
print("\n--- TOP 5 CANDIDATES ---")
for cid, change in top_5:
    print(f"ID: {cid} | Improvement: {-change:.4f}")

In [ ]:
import copy
import math
import torch
import numpy as np
from tqdm import tqdm
from torch_geometric.data import Data

def evaluate_candidate_node(cell_idx, cell_metadata_by_day, base_graphs, dataset, model, evaluation_indices):
    """
    Simulates adding a single candidate cell into the graph structure and calculates
    the prediction error strictly on the original observed stations.
    """
    mutated_graphs = {}
    
    # We need a fixed node list that matches the model's expected row order, PLUS the new node
    original_nodes = dataset.node_list
    mutated_node_list = original_nodes + [f"virtual_{cell_idx}"]
    
    # 1. Mutate graph topologies day-by-day for the specified evaluation period
    for idx in evaluation_indices:
        # Check both the window historical days and the target day
        window_dates = [dataset.dates[idx + t] for t in range(dataset.window_size + 1)]
        
        for d in window_dates:
            if d in mutated_graphs:
                continue
                
            # Create a deep copy of the day's original wind graph
            G = copy.deepcopy(base_graphs[d])
            cell_data = cell_metadata_by_day[d][cell_idx]
            
            # Add the candidate cell as a virtual node with its transformed anomaly features
            v_id = cell_data['cell_id']
            G.add_node(v_id,
                       station_lat=cell_data['lat'],
                       station_lon=cell_data['lon'],
                       wind_dir=cell_data['wind_dir'],
                       wind_speed=cell_data['wind_speed'],
                       features=cell_data['features'])
            
            # Calculate new directed edges from existing nodes to virtual, and virtual to existing
            for node in original_nodes:
                ndata = G.nodes[node]
                
                # Check Distance (Existing -> Virtual)
                dist_out = haversine_km(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
                if dist_out <= DIST_THRESHOLD_KM:
                    bearing = bearing_deg(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
                    diff = angle_diff_deg(ndata['wind_dir'], bearing)
                    score = max(math.cos(math.radians(diff)), 0.0) * ndata['wind_speed']
                    if score > SCORE_THRESHOLD:
                        G.add_edge(node, v_id, weight=score, distance_km=dist_out, angle_diff_deg=diff)
                        
                # Check Distance (Virtual -> Existing)
                dist_in = haversine_km(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
                if dist_in <= DIST_THRESHOLD_KM:
                    bearing = bearing_deg(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
                    diff = angle_diff_deg(cell_data['wind_dir'], bearing)
                    score = max(math.cos(math.radians(diff)), 0.0) * cell_data['wind_speed']
                    if score > SCORE_THRESHOLD:
                        G.add_edge(v_id, node, weight=score, distance_km=dist_in, angle_diff_deg=diff)
                        
            mutated_graphs[d] = G

    # 2. Run forward pass through the model using mutated sequences
    model.eval()
    total_mse = 0.0
    valid_days = 0
    
    with torch.no_grad():
        for idx in evaluation_indices:
            target_date = dataset.dates[idx + dataset.window_size]
            window_dates = dataset.dates[idx : idx + dataset.window_size]
            
            # Construct feature matrix tracking the new node order
            seq_feats = []
            for d in window_dates:
                g = mutated_graphs[d]
                day_feats = [g.nodes[node]['features'] for node in mutated_node_list]
                seq_feats.append(day_feats)
            X = torch.tensor(seq_feats, dtype=torch.float).permute(1, 0, 2)
            
            # Target alignment: original stations have true values, virtual node has NaN
            y_list = [dataset.pm25_dict[target_date].get(node, np.nan) for node in original_nodes] + [np.nan]
            y = torch.tensor(y_list, dtype=torch.float).unsqueeze(-1)
            
            # --- FIXED: MANUAL EXTRACTION BYPASSING torch_geometric.utils.from_networkx ---
            node_map = {node_id: i for i, node_id in enumerate(mutated_node_list)}
            edges_list = []
            weights_list = []

            current_g = mutated_graphs[target_date]
            for u, v, edata in current_g.edges(data=True):
                edges_list.append([node_map[u], node_map[v]])
                weights_list.append(edata.get('weight', 0.0))

            if len(edges_list) > 0:
                edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous()
                edge_weight = torch.tensor(weights_list, dtype=torch.float)
            else:
                edge_index = torch.empty((2, 0), dtype=torch.long)
                edge_weight = torch.empty((0,), dtype=torch.float)
            # ------------------------------------------------------------------------------
            
            # Forward pass
            out = model(X, edge_index, edge_weight)
            
            # CRITICAL: Mask evaluates error strictly on the original physical nodes
            mask = ~torch.isnan(y)
            if mask.sum() > 0:
                total_mse += torch.mean((out[mask] - y[mask]) ** 2).item()
                valid_days += 1
                
    return total_mse / valid_days if valid_days > 0 else float('inf')


# --------------------------------------------------------------------
# RUN THE OPTIMIZATION SEARCH LOOP
# --------------------------------------------------------------------
grid_cell_indices = list(processed_grid_by_day[dates[0]].keys())
grid_perf_results = {}

print("--- STARTING CANDIDATE SEARCH OPTIMIZATION (TRAIN/VAL PERIOD) ---")
optimization_period = val_indices 

for cell_idx in tqdm(grid_cell_indices, desc="Evaluating Grid Positions"):
    simulated_mse = evaluate_candidate_node(
        cell_idx=cell_idx,
        cell_metadata_by_day=processed_grid_by_day,
        base_graphs=graphs,
        dataset=dataset,
        model=trained_model,
        evaluation_indices=optimization_period
    )
    grid_perf_results[cell_idx] = simulated_mse

# Identify the absolute best candidate cell index
best_cell_idx = min(grid_perf_results, key=grid_perf_results.get)
print(f"\nOptimization Complete!")
print(f"Optimal Candidate Location found at Grid Cell ID: virtual_{best_cell_idx}")
print(f"Minimized Internal Station MSE: {grid_perf_results[best_cell_idx]:.4f}")


# --------------------------------------------------------------------
# RUN THE VALIDATION STEP ON THE UNSEEN TEST SET
# --------------------------------------------------------------------
print(f"\n--- VALIDATING CELL virtual_{best_cell_idx} ON FUTURE TEST DATA ---")

optimal_location_test_mse = evaluate_candidate_node(
    cell_idx=best_cell_idx,
    cell_metadata_by_day=processed_grid_by_day,
    base_graphs=graphs,
    dataset=dataset,
    model=trained_model,
    evaluation_indices=test_indices
)

print("\n================ FINAL INFILL EVALUATION ================")
print(f"Original Baseline Network Test MSE : {final_mse:.4f}")
print(f"Mutated Infill Network Test MSE     : {optimal_location_test_mse:.4f}")
print("-" * 57)

improvement = final_mse - optimal_location_test_mse
if improvement > 0:
    print(f"🚀 SUCCESS: Adding this cell reduced station error by {improvement:.4f} ({ (improvement/final_mse)*100 :.2f}%)")
    print("This confirms the node catches upwind advection structures that were missing.")
else:
    print("⚠️ No Improvement: The candidate node did not successfully reduce errors on future data.")
    print("This indicates spatial feature redundancy or a regime shift in wind direction over the test period.")
print("=========================================================")

In [ ]:
#the next approach does this- 

# =========================================================================================
# SENSOR INFILL OPTIMIZATION FRAMEWORK: OPTIONS A & B
# =========================================================================================
# This framework evaluates which candidate grid cell adds the most value to our 
# air quality sensor network, using two complementary strategies to ensure real-world success:
#
# OPTION A: MULTI-SEASON REPRESENTATIVE SCREENING
# -----------------------------------------------
# Instead of testing candidates on a single continuous block of time, we sample evaluation 
# days evenly across our entire historical timeline. This captures multiple distinct weather 
# and wind regimes (e.g., winter northwest winds vs. summer monsoons in Seoul). 
# By measuring how well a candidate cell routes meteorological information across these 
# diverse periods, we filter out "one-hit wonders" and isolate the Top 3 most resilient 
# locations that provide stable, year-round upwind coverage.
#
# OPTION B: CANDIDATE-INFORMED MODEL RE-TRAINING
# ----------------------------------------------
# A frozen model cannot automatically understand how to weight messages coming from a brand-new 
# node. To fairly evaluate our top 3 candidates, we permanently inject each virtual node into 
# the graph structure and re-train a fresh Spatio-Temporal GCN from scratch for each setup. 
# This allows the neural network to explicitly learn the unique temporal weather trends 
# and wind-transport pathways introduced by that specific candidate. 
#
# THE FINAL TEST:
# ---------------
# We evaluate these freshly trained models on a completely unseen, future test set. 
# If a model achieves a lower prediction error (MSE) on our original physical stations than 
# the baseline network did, it proves that monitoring weather at that specific grid coordinate 
# fundamentally solves spatial feature homogeneity and optimizes the network's footprint.
# =========================================================================================

In [ ]:
import numpy as np
import torch
import copy
from tqdm import tqdm

# --------------------------------------------------------------------
# SETUP: DESIGNING MULTI-SEASON REPRESENTATIVE INDEXES (OPTION A)
# --------------------------------------------------------------------
# Instead of a single block, we sample indices uniformly across the 
# historical training/validation timeline to capture varied wind regimes.
total_seqs = len(dataset)
train_val_end_idx = int(total_seqs * 0.80)  # Reserve the last 20% strictly for future testing

# Sample 40 days spread evenly across the history to represent multiple seasons
num_representative_days = min(40, train_val_end_idx)
multi_season_indices = np.linspace(0, train_val_end_idx - 1, num_representative_days, dtype=int).tolist()

# Define your clean future test set (e.g., the final seasonal block)
future_test_indices = list(range(train_val_end_idx, total_seqs))

print(f"--- RE-OPTIMIZING WITH MULTI-SEASON FOOTPRINT (OPTION A) ---")
print(f"Evaluating candidates over {len(multi_season_indices)} distinct days across seasons.")

# Reuse our manual evaluation logic to check all grid positions
grid_cell_indices = list(processed_grid_by_day[dates[0]].keys())
multi_season_results = {}

for cell_idx in tqdm(grid_cell_indices, desc="Multi-Season Screening"):
    simulated_mse = evaluate_candidate_node(
        cell_idx=cell_idx,
        cell_metadata_by_day=processed_grid_by_day,
        base_graphs=graphs,
        dataset=dataset,
        model=trained_model,  # Using baseline frozen model for initial screening
        evaluation_indices=multi_season_indices
    )
    multi_season_results[cell_idx] = simulated_mse

# Identify Top 3 Candidate Locations that are resilient across seasons
sorted_candidates = sorted(multi_season_results.items(), key=lambda x: x[1])
top_3_candidates = [cell_idx for cell_idx, mse in sorted_candidates[:3]]

print("\nTop 3 Candidate Locations Found:")
for i, c_idx in enumerate(top_3_candidates):
    print(f"  Rank {i+1}: Grid Cell ID 'virtual_{c_idx}' (Screening MSE: {multi_season_results[c_idx]:.4f})")


# --------------------------------------------------------------------
# CONFIGURATION: CANDIDATE-INFORMED RE-TRAINING ENGINE (OPTION B)
# --------------------------------------------------------------------
# A custom dataset subclass that hardcodes a specific candidate cell into 
# every single daily sequence, allowing a brand new model to train on it.
class MutatedSpatioTemporalDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, cell_idx, cell_metadata_by_day, base_graphs):
        self.base_dataset = base_dataset
        self.cell_idx = cell_idx
        self.cell_metadata = cell_metadata_by_day
        self.base_graphs = base_graphs
        
        # --- ADD THIS LINE TO FIX THE ATTRIBUTE ERROR ---
        self.window_size = base_dataset.window_size
        # ------------------------------------------------
        
        # New fixed node order explicitly including the virtual location
        self.original_nodes = base_dataset.node_list
        self.mutated_node_list = self.original_nodes + [f"virtual_{cell_idx}"]
        
    def __len__(self):
        return len(self.base_dataset)
        
    def __getitem__(self, idx):
        target_date = self.base_dataset.dates[idx + self.base_dataset.window_size]
        window_dates = self.base_dataset.dates[idx : idx + self.base_dataset.window_size]
        
        # 1. Rebuild topologies on the fly with the virtual node injected
        mutated_graphs = {}
        for d in window_dates + [target_date]:
            G = copy.deepcopy(self.base_graphs[d])
            cell_data = self.cell_metadata[d][self.cell_idx]
            v_id = cell_data['cell_id']
            
            G.add_node(v_id, station_lat=cell_data['lat'], station_lon=cell_data['lon'],
                       wind_dir=cell_data['wind_dir'], wind_speed=cell_data['wind_speed'], features=cell_data['features'])
            
            # Form wind-driven edges to and from the new node
            for node in self.original_nodes:
                ndata = G.nodes[node]
                # Outgoing from existing to virtual
                dist_out = haversine_km(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
                if dist_out <= DIST_THRESHOLD_KM:
                    score = max(math.cos(math.radians(angle_diff_deg(ndata['wind_dir'], bearing_deg(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])))), 0.0) * ndata['wind_speed']
                    if score > SCORE_THRESHOLD: G.add_edge(node, v_id, weight=score)
                # Incoming from virtual to existing
                dist_in = haversine_km(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
                if dist_in <= DIST_THRESHOLD_KM:
                    score = max(math.cos(math.radians(angle_diff_deg(cell_data['wind_dir'], bearing_deg(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])))), 0.0) * cell_data['wind_speed']
                    if score > SCORE_THRESHOLD: G.add_edge(v_id, node, weight=score)
            mutated_graphs[d] = G
            
        # 2. Extract feature sequence
        seq_feats = [[mutated_graphs[d].nodes[node]['features'] for node in self.mutated_node_list] for d in window_dates]
        X = torch.tensor(seq_feats, dtype=torch.float).permute(1, 0, 2)
        
        # 3. Setup targets (virtual node target remains NaN)
        y_list = [self.base_dataset.pm25_dict[target_date].get(node, np.nan) for node in self.original_nodes] + [np.nan]
        y = torch.tensor(y_list, dtype=torch.float).unsqueeze(-1)
        
        # 4. Map edges manually
        node_map = {node_id: i for i, node_id in enumerate(self.mutated_node_list)}
        edges_list, weights_list = [], []
        for u, v, edata in mutated_graphs[target_date].edges(data=True):
            edges_list.append([node_map[u], node_map[v]])
            weights_list.append(edata.get('weight', 0.0))
            
        edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous() if edges_list else torch.empty((2, 0), dtype=torch.long)
        edge_weight = torch.tensor(weights_list, dtype=torch.float) if weights_list else torch.empty((0,), dtype=torch.float)
        
        return Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)


# --------------------------------------------------------------------
# EXECUTION: RUNNING RE-TRAINING PIPELINES (OPTION B)
# --------------------------------------------------------------------
print(f"\n--- STARTING OPTION B: RE-TRAINING TOP CANDIDATES FROM SCRATCH ---")

# Train/Val index splits for the re-training phase
train_end_idx = int(train_val_end_idx * 0.80)
sub_train_indices = list(range(0, train_end_idx))
sub_val_indices = list(range(train_end_idx, train_val_end_idx))

final_test_results = {}

for rank, cell_idx in enumerate(top_3_candidates):
    print(f"\n[Evaluating Rank {rank+1}] Re-training Spatio-Temporal GCN with 'virtual_{cell_idx}' built-in...")
    
    # Instantiate the modified dataset containing this specific candidate cell
    mutated_dataset = MutatedSpatioTemporalDataset(
        base_dataset=dataset,
        cell_idx=cell_idx,
        cell_metadata_by_day=processed_grid_by_day,
        base_graphs=graphs
    )
    
    # Train the fresh model architecture on the mutated timeline
    fresh_model = train_baseline_model(
        dataset=mutated_dataset,
        train_indices=sub_train_indices,
        val_indices=sub_val_indices,
        epochs=20,  # 20 epochs per candidate keeps this highly efficient
        lr=0.001
    )
    
    # Evaluate the fully-trained model on the completely unseen future test set
    fresh_model.eval()
    test_set = [mutated_dataset[i] for i in future_test_indices]
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
    
    test_predictions, test_ground_truth = [], []
    with torch.no_grad():
        for batch in test_loader:
            out = fresh_model(batch.x, batch.edge_index, batch.edge_attr)
            # Match strictly against the original physical sensors
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                test_predictions.extend(out[mask].cpu().numpy().flatten())
                test_ground_truth.extend(batch.y[mask].cpu().numpy().flatten())
                
    cand_test_mse = np.mean((np.array(test_predictions) - np.array(test_ground_truth)) ** 2)
    final_test_results[cell_idx] = cand_test_mse
    print(f"➔ Candidate 'virtual_{cell_idx}' Final Test MSE: {cand_test_mse:.4f}")

# --------------------------------------------------------------------
# FINAL BREAKDOWN REPORT
# --------------------------------------------------------------------
print("\n================== GLOBAL INFILL PERFORMANCE REPORT ==================")
print(f"Original Benchmark Network Test MSE : {final_mse:.4f}")
print("-" * 70)

for rank, cell_idx in enumerate(top_3_candidates):
    cand_mse = final_test_results[cell_idx]
    net_change = final_mse - cand_mse
    if net_change > 0:
        status = f"🚀 IMPROVEMENT: Reduced error by {net_change:.4f} ({(net_change/final_mse)*100:.2f}%)"
    else:
        status = f"⚠️ DEGRADED   : Increased error by {abs(net_change):.4f} ({(abs(net_change)/final_mse)*100:.2f}%)"
    print(f"Rank {rank+1} (Cell {cell_idx:<4}) | Test MSE: {cand_mse:.4f} | {status}")
print("======================================================================")

In [ ]:
import torch
import numpy as np
import copy
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from tqdm import tqdm

# 1. DATASET DEFINITION (Option B)
class MutatedSpatioTemporalDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, cell_idx, cell_metadata_by_day, base_graphs):
        self.base_dataset = base_dataset
        self.cell_idx = cell_idx
        self.cell_metadata = cell_metadata_by_day
        self.base_graphs = base_graphs
        self.window_size = base_dataset.window_size
        self.original_nodes = base_dataset.node_list
        self.mutated_node_list = self.original_nodes + [f"virtual_{cell_idx}"]
        
    def __len__(self):
        return len(self.base_dataset)
        
    def __getitem__(self, idx):
        target_date = self.base_dataset.dates[idx + self.window_size]
        window_dates = self.base_dataset.dates[idx : idx + self.window_size]
        
        mutated_graphs = {}
        for d in window_dates + [target_date]:
            G = copy.deepcopy(self.base_graphs[d])
            cell_data = self.cell_metadata[d][self.cell_idx]
            v_id = cell_data['cell_id']
            
            G.add_node(v_id, station_lat=cell_data['lat'], station_lon=cell_data['lon'],
                       wind_dir=cell_data['wind_dir'], wind_speed=cell_data['wind_speed'], 
                       features=cell_data['features'])
            
            for node in self.original_nodes:
                ndata = G.nodes[node]
                # Logic: Outgoing/Incoming edges based on threshold
                dist_out = haversine_km(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
                if dist_out <= DIST_THRESHOLD_KM:
                    score = max(np.cos(np.radians(angle_diff_deg(ndata['wind_dir'], bearing_deg(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])))), 0.0) * ndata['wind_speed']
                    if score > SCORE_THRESHOLD: G.add_edge(node, v_id, weight=score)
                
                dist_in = haversine_km(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
                if dist_in <= DIST_THRESHOLD_KM:
                    score = max(np.cos(np.radians(angle_diff_deg(cell_data['wind_dir'], bearing_deg(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])))), 0.0) * cell_data['wind_speed']
                    if score > SCORE_THRESHOLD: G.add_edge(v_id, node, weight=score)
            mutated_graphs[d] = G
            
        seq_feats = [[mutated_graphs[d].nodes[node]['features'] for node in self.mutated_node_list] for d in window_dates]
        X = torch.tensor(seq_feats, dtype=torch.float).permute(1, 0, 2)
        y_list = [self.base_dataset.pm25_dict[target_date].get(node, np.nan) for node in self.original_nodes] + [np.nan]
        y = torch.tensor(y_list, dtype=torch.float).unsqueeze(-1)
        
        node_map = {node_id: i for i, node_id in enumerate(self.mutated_node_list)}
        edges_list, weights_list = [], []
        for u, v, edata in mutated_graphs[target_date].edges(data=True):
            edges_list.append([node_map[u], node_map[v]])
            weights_list.append(edata.get('weight', 0.0))
            
        edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous() if edges_list else torch.empty((2, 0), dtype=torch.long)
        edge_weight = torch.tensor(weights_list, dtype=torch.float) if weights_list else torch.empty((0,), dtype=torch.float)
        
        return Data(x=X, edge_index=edge_index, edge_attr=edge_weight, y=y)

# 2. BASELINE EVALUATOR
def evaluate_loocv_idw_on_indices(dataset, indices, power=2.0):
    mse_list = []
    for idx in indices:
        data = dataset[idx]
        # Assume coordinates are in the last dimension of the input feature tensor
        coords = data.x[:, -1, -2:] 
        targets = data.y.squeeze()
        num_nodes = coords.shape[0]
        mask = ~torch.isnan(targets)
        if mask.sum() < 2: continue
        dist_matrix = torch.cdist(coords, coords, p=2)
        inv_dist = 1.0 / (dist_matrix ** power + 1e-9)
        for i in range(num_nodes):
            if not mask[i]: continue
            neighbor_mask = mask.clone()
            neighbor_mask[i] = False
            weights = inv_dist[i, neighbor_mask]
            vals = targets[neighbor_mask]
            if weights.sum() > 0:
                prediction = torch.sum(weights * vals) / torch.sum(weights)
                mse_list.append((prediction - targets[i]) ** 2)
    return np.mean(mse_list) if mse_list else 0.0

# 3. EXECUTION PIPELINE
print(f"\n--- STARTING OPTION B: RE-TRAINING TOP CANDIDATES ---")
final_test_results = {}
train_end_idx = int(train_val_end_idx * 0.80)
sub_train_indices = list(range(0, train_end_idx))
sub_val_indices = list(range(train_end_idx, train_val_end_idx))

for rank, cell_idx in enumerate(top_3_candidates):
    print(f"\n[Evaluating Rank {rank+1}] Re-training with 'virtual_{cell_idx}'...")
    mutated_dataset = MutatedSpatioTemporalDataset(dataset, cell_idx, processed_grid_by_day, graphs)
    
    fresh_model = train_baseline_model(
        dataset=mutated_dataset,
        train_indices=sub_train_indices,
        val_indices=sub_val_indices,
        epochs=20,
        lr=0.001
    )
    
    fresh_model.eval()
    test_set = [mutated_dataset[i] for i in future_test_indices]
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
    
    test_predictions, test_ground_truth = [], []
    with torch.no_grad():
        for batch in test_loader:
            out = fresh_model(batch.x, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                test_predictions.extend(out[mask].cpu().numpy().flatten())
                test_ground_truth.extend(batch.y[mask].cpu().numpy().flatten())
                
    cand_test_mse = np.mean((np.array(test_predictions) - np.array(test_ground_truth)) ** 2)
    final_test_results[cell_idx] = cand_test_mse
    print(f"➔ Candidate 'virtual_{cell_idx}' Final Test MSE: {cand_test_mse:.4f}")

# 4. FINAL BREAKDOWN
print("\n" + "="*70)
print("COMPUTING LOOCV BASELINE...")
baseline_loocv_mse = evaluate_loocv_idw_on_indices(dataset, future_test_indices)
print(f"Sensor-Only LOOCV Baseline MSE : {baseline_loocv_mse:.4f}")
print("-" * 70)

for rank, cell_idx in enumerate(top_3_candidates):
    cand_mse = final_test_results[cell_idx]
    loocv_gain = baseline_loocv_mse - cand_mse
    status = f"🚀 BEATS BASELINE: +{(loocv_gain/baseline_loocv_mse)*100:.1f}%" if loocv_gain > 0 else f"⚠️ UNDERPERFORMS: {(loocv_gain/baseline_loocv_mse)*100:.1f}%"
    print(f"Rank {rank+1} (Cell {cell_idx:<4}) | Test MSE: {cand_mse:.4f} | {status}")
print("="*70)

In [ ]:
# Run this on your standard dataset (no virtual node)
original_model = train_baseline_model(dataset=dataset, ...) # Your original setup
baseline_mse = evaluate_loocv_idw_on_indices(dataset, future_test_indices)
model_mse = get_errors(original_model, standard_loader)

print(f"IDW Baseline: {baseline_mse:.4f}")
print(f"GNN Model   : {model_mse:.4f}")

In [ ]:
# Extract coordinates for your rank 1 winner
best_cell_id = 1 
sample_date = dates[0]

best_cell_coords = processed_grid_by_day[sample_date][best_cell_id]
new_station_lat = best_cell_coords['lat']
new_station_lon = best_cell_coords['lon']

print(f"🎯 Optimal New Sensor Placement Location:")
print(f"Latitude : {new_station_lat:.5f}")
print(f"Longitude: {new_station_lon:.5f}")

In [ ]:
import base64
import folium
from folium import plugins
from IPython.display import HTML
import math

# 1. Grab a clean date from your future test set to visualize
# (Using the very first index of your future test set)
test_date_key = dates[test_indices[0]]
print(f"Visualizing network footprint on future test date: {test_date_key}")

# Coordinates setup from BBOX
min_lon, min_lat, max_lon, max_lat = BBOX
center_lat = (min_lat + max_lat) / 2.0
center_lon = (min_lon + max_lon) / 2.0

# Extract our winning cell metadata for this specific test date
best_cell_id = 1  # Your 51.12% improvement winner
cell_data = processed_grid_by_day[test_date_key][best_cell_id]
v_id = cell_data['cell_id']

# 2. Build a fresh NetworkX graph for this day and inject the new optimal node
import copy
g_mutated = copy.deepcopy(graphs[test_date_key])

# Add the optimized virtual node
g_mutated.add_node(v_id,
                   station_lat=cell_data['lat'],
                   station_lon=cell_data['lon'],
                   wind_dir=cell_data['wind_dir'],
                   wind_speed=cell_data['wind_speed'])

# Calculate its dynamic wind edges for this specific day's wind vector
original_nodes = dataset.node_list
for node in original_nodes:
    ndata = g_mutated.nodes[node]
    
    # Outgoing connections (Existing -> New Virtual Node)
    dist_out = haversine_km(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
    if dist_out <= DIST_THRESHOLD_KM:
        bearing = bearing_deg(ndata['station_lat'], ndata['station_lon'], cell_data['lat'], cell_data['lon'])
        diff = angle_diff_deg(ndata['wind_dir'], bearing)
        score = max(math.cos(math.radians(diff)), 0.0) * ndata['wind_speed']
        if score > SCORE_THRESHOLD:
            g_mutated.add_edge(node, v_id, weight=score)
            
    # Incoming connections (New Virtual Node -> Existing)
    dist_in = haversine_km(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
    if dist_in <= DIST_THRESHOLD_KM:
        bearing = bearing_deg(cell_data['lat'], cell_data['lon'], ndata['station_lat'], ndata['station_lon'])
        diff = angle_diff_deg(cell_data['wind_dir'], bearing)
        score = max(math.cos(math.radians(diff)), 0.0) * cell_data['wind_speed']
        if score > SCORE_THRESHOLD:
            g_mutated.add_edge(v_id, node, weight=score)

# 3. Create the Folium Map object
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

# Draw BBOX domain bounding perimeter
folium.Rectangle(bounds=[[min_lat, min_lon], [max_lat, max_lon]], color='red', fill=False, weight=2, dash_array='5, 5').add_to(m)

# 4. Plot Nodes (Blue for physical sensors, Gold/Red Star for your new optimal location)
for node, data in g_mutated.nodes(data=True):
    if node == v_id:
        folium.Marker(
            location=[data['station_lat'], data['station_lon']],
            popup=f"<b>🏆 OPTIMAL INFILL NODE: {node}</b>",
            icon=folium.Icon(color='red', icon='star')
        ).add_to(m)
    else:
        folium.CircleMarker(
            location=[data['station_lat'], data['station_lon']],
            radius=6, color='black', fill=True, fill_color='dodgerblue', fill_opacity=0.9,
            popup=f"Sensor Station: {node}"
        ).add_to(m)

# 5. Plot Edges with Directional Arrow Tapes
for src, tgt, data in g_mutated.edges(data=True):
    udata = g_mutated.nodes[src]
    vdata = g_mutated.nodes[tgt]
    
    # Color link differently if it connects to our new optimal sensor to make it pop
    edge_color = 'crimson' if (src == v_id or tgt == v_id) else 'blue'
    
    line = folium.PolyLine(
        locations=[
            [udata['station_lat'], udata['station_lon']],
            [vdata['station_lat'], vdata['station_lon']]
        ],
        color=edge_color,
        weight=max(1.5, min(6, data.get('weight', 0) / 2)),
        opacity=0.7 if edge_color == 'blue' else 0.9,
    )
    line.add_to(m)
    
    # Single arrow in the middle to represent advection pathing direction
    plugins.PolyLineTextPath(
        line, '➤', repeat=False, center=True, offset=7, 
        attributes={'fill': edge_color, 'font-weight': 'bold', 'font-size': '14'}
    ).add_to(m)

# 6. Base64 Encode and render inside the isolated security sandbox frame
map_raw_html = m._repr_html_()
b64_html = base64.b64encode(map_raw_html.encode('utf-8')).decode('utf-8')
iframe_src = f"data:text/html;base64,{b64_html}"

# Wrap inside a clean display frame panel
panel = '<div style="border:1px solid #ccc; padding: 12px; background: #fff;">'
panel += f'<h3 style="margin:0 0 8px 0; font-family:sans-serif;">Network Footprint with Optimal Infill (Date: {test_date_key})</h3>'
panel += f'<iframe src="{iframe_src}" style="width:100%; height:600px; border:none;"></iframe>'
panel += '</div>'

display(HTML(panel))

In [ ]:
#statistical significance. 
import numpy as np
import torch
from scipy import stats
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

print("--- RUNNING MODEL SIGNIFICANCE EXPERIMENT (BASELINE VS INFILL) ---")

# Setup the clean test data loader for BOTH models
future_test_set_base = [dataset[i] for i in future_test_indices]
future_test_set_infill = [mutated_dataset[i] for i in future_test_indices]

test_loader_base = DataLoader(future_test_set_base, batch_size=1, shuffle=False)
test_loader_infill = DataLoader(future_test_set_infill, batch_size=1, shuffle=False)

# Track errors day-by-day
daily_mse_baseline = []
daily_mse_infill = []

# 1. Collect Daily Errors for the Baseline Model
trained_model.eval()
with torch.no_grad():
    for batch in test_loader_base:
        out = trained_model(batch.x, batch.edge_index, batch.edge_attr)
        mask = ~torch.isnan(batch.y)
        if mask.sum() > 0:
            day_mse = torch.mean((out[mask] - batch.y[mask]) ** 2).item()
            daily_mse_baseline.append(day_mse)

# 2. Collect Daily Errors for the Infill Model (Cell 1 built-in)
fresh_model.eval() # This is your model trained with Cell 1 from Option B
with torch.no_grad():
    for batch in test_loader_infill:
        out = fresh_model(batch.x, batch.edge_index, batch.edge_attr)
        # Mask evaluates strictly on the original physical nodes for a fair comparison
        mask = ~torch.isnan(batch.y)
        if mask.sum() > 0:
            day_mse = torch.mean((out[mask] - batch.y[mask]) ** 2).item()
            daily_mse_infill.append(day_mse)

daily_mse_baseline = np.array(daily_mse_baseline)
daily_mse_infill = np.array(daily_mse_infill)

# 3. Calculate Paired t-test
t_stat, p_value = stats.ttest_rel(daily_mse_infill, daily_mse_baseline, alternative='less')

print("\n================ STATISTICAL SIGNIFICANCE SUMMARY ================")
print(f"Total Test Days Evaluated       : {len(daily_mse_baseline)}")
print(f"Baseline Model Avg Daily MSE    : {np.mean(daily_mse_baseline):.4f}")
print(f"Infill Model Avg Daily MSE      : {np.mean(daily_mse_infill):.4f}")
print(f"Average Daily Error Reduction   : {np.mean(daily_mse_baseline - daily_mse_infill):.4f}")
print("-" * 66)
print(f"Calculated t-statistic          : {t_stat:.4f}")
print(f"One-tailed p-value              : {p_value:.6f}")
print("-" * 66)

if p_value < 0.05:
    print(f"🚀 STATISTICALLY SIGNIFICANT (p = {p_value:.6f}): Reject the null hypothesis!")
    print("The infill model consistently and reliably outperforms the baseline model across")
    print("the test timeline. The added wind-driven spatial tracking is physically meaningful.")
else:
    print(f"⚠️ NOT SIGNIFICANT (p = {p_value:.6f}): Fail to reject the null hypothesis.")
    print("The error reduction is not uniform across the test timeline and could be due to chance.")
print("==================================================================")

# 4. Plot Daily Error Comparison
plt.figure(figsize=(10, 5))
plt.plot(daily_mse_baseline, label='Baseline Model (No Infill)', color='gray', alpha=0.6, linestyle='--')
plt.plot(daily_mse_infill, label='Infill Model (With Cell 1)', color='crimson', alpha=0.8)
plt.fill_between(range(len(daily_mse_baseline)), daily_mse_baseline, daily_mse_infill, 
                 where=(daily_mse_baseline > daily_mse_infill), facecolor='green', alpha=0.2, label='Infill Advantage')
plt.title('Day-by-Day Test MSE Comparison')
plt.xlabel('Test Days (Chronological)')
plt.ylabel('Mean Squared Error (MSE)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#checking if this location is optimal for different seasons. 

import numpy as np
import torch
import copy
from sklearn.model_selection import KFold
from torch_geometric.loader import DataLoader
from scipy import stats

# --------------------------------------------------------------------
# SETUP: CROSS-SEASONAL K-FOLD SPLITTER
# --------------------------------------------------------------------
total_sequences = len(dataset)
all_indices = np.arange(total_sequences)

# We use 5 folds to ensure substantial seasonal representation in each slice
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_baseline_mses = []
fold_infill_mses = []

print(f"--- STARTING CROSS-SEASONAL K-FOLD EVALUATION ---")
print(f"Total historical sequences: {total_sequences} | Running 5 distinct seasonal folds.\n")

# Instantiate the all-season mutated dataset for your winning node (Cell 1)
mutated_dataset = MutatedSpatioTemporalDataset(
    base_dataset=dataset,
    cell_idx=1,
    cell_metadata_by_day=processed_grid_by_day,
    base_graphs=graphs
)

# --------------------------------------------------------------------
# ITERATING THROUGH THE SEASONS: THE FOLD LOOP
# --------------------------------------------------------------------
for fold, (train_idx, test_idx) in enumerate(kf.split(all_indices)):
    print(f"⚡ Processing Fold {fold + 1}/5...")
    
    # Sub-split training into train/validation sets (80/20) within the fold
    val_split_point = int(len(train_idx) * 0.8)
    fold_train_idx = train_idx[:val_split_point].tolist()
    fold_val_idx = train_idx[val_split_point:].tolist()
    fold_test_idx = test_idx.tolist()
    
    # 1. Train Baseline Model from scratch for this fold's seasonal mix
    print(f"   ↳ Training Baseline Model...")
    fold_baseline_model = train_baseline_model(
        dataset=dataset,
        train_indices=fold_train_idx,
        val_indices=fold_val_idx,
        epochs=15,  # 15 epochs per fold keeps this framework highly efficient
        lr=0.001
    )
    
    # 2. Train Infill Model from scratch for this fold's seasonal mix
    print(f"   ↳ Training Infill Model (With Cell 1)...")
    fold_infill_model = train_baseline_model(
        dataset=mutated_dataset,
        train_indices=fold_train_idx,
        val_indices=fold_val_idx,
        epochs=15,
        lr=0.001
    )
    
    # 3. Evaluate both models on the completely unseen Fold Test Set
    fold_test_set_base = [dataset[i] for i in fold_test_idx]
    fold_test_set_infill = [mutated_dataset[i] for i in fold_test_idx]
    
    loader_base = DataLoader(fold_test_set_base, batch_size=1, shuffle=False)
    loader_infill = DataLoader(fold_test_set_infill, batch_size=1, shuffle=False)
    
    # Evaluate Baseline
    fold_baseline_model.eval()
    base_errors = []
    with torch.no_grad():
        for batch in loader_base:
            out = fold_baseline_model(batch.x, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                base_errors.append(torch.mean((out[mask] - batch.y[mask]) ** 2).item())
                
    # Evaluate Infill
    fold_infill_model.eval()
    infill_errors = []
    with torch.no_grad():
        for batch in loader_infill:
            out = fold_infill_model(batch.x, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                infill_errors.append(torch.mean((out[mask] - batch.y[mask]) ** 2).item())
                
    fold_base_mse = np.mean(base_errors)
    fold_inf_mse = np.mean(infill_errors)
    
    fold_baseline_mses.append(fold_base_mse)
    fold_infill_mses.append(fold_inf_mse)
    
    improvement = ((fold_base_mse - fold_inf_mse) / fold_base_mse) * 100
    print(f"   ➔ Fold {fold+1} Complete | Baseline MSE: {fold_base_mse:.2f} | Infill MSE: {fold_inf_mse:.2f} | Gain: {improvement:.2f}%")

# --------------------------------------------------------------------
# FINAL METRIC & PLOTS ACROSS FOLDS
# --------------------------------------------------------------------
fold_baseline_mses = np.array(fold_baseline_mses)
fold_infill_mses = np.array(fold_infill_mses)

# Execute cross-fold paired t-test
t_stat_kf, p_val_kf = stats.ttest_rel(fold_infill_mses, fold_baseline_mses, alternative='less')

print("\n================ FINAL K-FOLD ROBUSTNESS REPORT ================")
print(f"Grand Mean Baseline Cross-Val MSE : {np.mean(fold_baseline_mses):.4f}")
print(f"Grand Mean Infill Cross-Val MSE   : {np.mean(fold_infill_mses):.4f}")
print(f"Overall Cross-Seasonal Reduction   : {np.mean(fold_baseline_mses - fold_infill_mses):.4f}")
print("-" * 64)
print(f"Cross-Seasonal t-statistic        : {t_stat_kf:.4f}")
print(f"Cross-Seasonal p-value            : {p_val_kf:.6f}")
print("-" * 64)

if p_val_kf < 0.05:
    print(f"🚀 ROBUST ACCEPTANCE (p = {p_val_kf:.6f}): Reject H0!")
    print("When trained across all changing seasons, the network architectures containing")
    print("Cell 1 display a resilient, uniform error reduction regardless of the time of year.")
else:
    print(f"⚠️ REJECTED (p = {p_val_kf:.6f}): Fail to reject H0.")
    print("The error profile indicates that node location advantage depends entirely on individual seasons.")
print("=================================================================")

In [ ]:
#the following code uses seasonal data, ie check our results by season.

In [ ]:
# Define your distinct chronological atmospheric blocks
seasonal_blocks = {
    'Winter_Base' : ['2024-12', '2025-01'], # Data we know right now
    'Spring_Target': ['2025-03', '2025-04'], # The upcoming season we want to optimize for
    
    'Spring_Base' : ['2025-04', '2025-05'],
    'Summer_Target': ['2025-06', '2025-07']
}

def get_indices_for_months(month_list):
    idx_list = []
    for i, date_str in enumerate(dataset.dates):
        if date_str.startswith(tuple(month_list)):
            if i >= dataset.window_size and i < len(dataset):
                idx_list.append(i - dataset.window_size)
    return idx_list

In [ ]:
results_by_season = {}

# Let's test the Winter -> Spring transition as an example
base_months = seasonal_blocks['Winter_Base']
target_months = seasonal_blocks['Spring_Target']

base_indices = get_indices_for_months(base_months)
target_indices = get_indices_for_months(target_months)

print(f"--- RUNNING PROCEEDING-SEASON OPTIMIZATION ---")
print(f"Screening candidates using history: {base_months}")
print(f"Evaluating chosen node on upcoming season: {target_months}\n")

# 1. Screen candidates strictly on the lookback history
historical_screening = {}
for cell_idx in tqdm(grid_cell_indices, desc="Seasonal Screening"):
    simulated_mse = evaluate_candidate_node(
        cell_idx=cell_idx, cell_metadata_by_day=processed_grid_by_day,
        base_graphs=graphs, dataset=dataset, model=trained_model,
        evaluation_indices=base_indices
    )
    historical_screening[cell_idx] = simulated_mse

# Identify the localized optimal node for this specific transition
seasonal_winner_idx = min(historical_screening, key=historical_screening.get)
print(f"\n🎯 Optimal Node found for this regime: 'virtual_{seasonal_winner_idx}'")

# 2. Build the mutated dataset using the season-specific winner
seasonal_mutated_dataset = MutatedSpatioTemporalDataset(
    base_dataset=dataset, cell_idx=seasonal_winner_idx,
    cell_metadata_by_day=processed_grid_by_day, base_graphs=graphs
)

# 3. Train from scratch on the history, letting it learn this season's pathways
print(f"\nRe-training model with tailored node 'virtual_{seasonal_winner_idx}'...")
seasonal_model = train_baseline_model(
    dataset=seasonal_mutated_dataset,
    train_indices=base_indices,
    val_indices=base_indices[-10:], # use tail end for validation
    epochs=15, lr=0.001
)

# 4. Evaluate on the PROCEEDING target season
seasonal_model.eval()
target_loader = DataLoader([seasonal_mutated_dataset[i] for i in target_indices], batch_size=1, shuffle=False)

target_preds, target_true = [], []
with torch.no_grad():
    for batch in target_loader:
        out = seasonal_model(batch.x, batch.edge_index, batch.edge_attr)
        mask = ~torch.isnan(batch.y)
        if mask.sum() > 0:
            target_preds.extend(out[mask].cpu().numpy().flatten())
            target_true.extend(batch.y[mask].cpu().numpy().flatten())

proceeding_mse = np.mean((np.array(target_preds) - np.array(target_true)) ** 2)
print(f"\n➔ Final Result on Unseen Proceeding Season Test Set: {proceeding_mse:.4f}")

In [ ]:
# Define the sequential rolling horizon blocks across your 9 months of data


#this defines different seasons and checks how adding a node based on one seasons data affects the next.

#we find migration of optimal node. 
rolling_windows = [
    {
        "name": "Winter to Spring Transition",
        "lookback_months": ["2024-12", "2025-01"],
        "proceeding_months": ["2025-03", "2025-04"]
    },
    {
        "name": "Spring to Summer Transition",
        "lookback_months": ["2025-03", "2025-04"],
        "proceeding_months": ["2025-06", "2025-07"]
    },
    {
        "name": "Summer to Late-Summer Transition",
        "lookback_months": ["2025-05", "2025-06"],
        "proceeding_months": ["2025-07", "2025-08"]
    }
]

migration_summary = []

print("--- RUNNING GLOBAL ROLLING HORIZON MIGRATION ENGINE ---")

for window in rolling_windows:
    print(f"\n🚀 Evaluating: {window['name']}")
    
    base_indices = get_indices_for_months(window['lookback_months'])
    target_indices = get_indices_for_months(window['proceeding_months'])
    
    # 1. Screen candidates strictly on the lookback history
    historical_screening = {}
    for cell_idx in grid_cell_indices:
        simulated_mse = evaluate_candidate_node(
            cell_idx=cell_idx, cell_metadata_by_day=processed_grid_by_day,
            base_graphs=graphs, dataset=dataset, model=trained_model,
            evaluation_indices=base_indices
        )
        historical_screening[cell_idx] = simulated_mse
    
    # Identify the localized optimal node for this specific transition
    seasonal_winner_idx = min(historical_screening, key=historical_screening.get)
    coords = processed_grid_by_day[dates[0]][seasonal_winner_idx]
    
    # 2. Get baseline performance on target window for benchmarking
    # Evaluate original frozen model on the target window
    base_preds, base_true = [], []
    trained_model.eval()
    target_set_base = [dataset[i] for i in target_indices]
    loader_base = DataLoader(target_set_base, batch_size=1, shuffle=False)
    
    with torch.no_grad():
        for batch in loader_base:
            out = trained_model(batch.x, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                base_preds.extend(out[mask].cpu().numpy().flatten())
                base_true.extend(batch.y[mask].cpu().numpy().flatten())
    baseline_target_mse = np.mean((np.array(base_preds) - np.array(base_true)) ** 2)
    
    # 3. Train from scratch on the history with the tailored node built-in
    seasonal_mutated_dataset = MutatedSpatioTemporalDataset(
        base_dataset=dataset, cell_idx=seasonal_winner_idx,
        cell_metadata_by_day=processed_grid_by_day, base_graphs=graphs
    )
    
    seasonal_model = train_baseline_model(
        dataset=seasonal_mutated_dataset,
        train_indices=base_indices,
        val_indices=base_indices[-10:],
        epochs=15, lr=0.001
    )
    
    # 4. Evaluate on the PROCEEDING target season
    seasonal_model.eval()
    target_set_inf = [seasonal_mutated_dataset[i] for i in target_indices]
    loader_inf = DataLoader(target_set_inf, batch_size=1, shuffle=False)
    
    target_preds, target_true = [], []
    with torch.no_grad():
        for batch in loader_inf:
            out = seasonal_model(batch.x, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            if mask.sum() > 0:
                target_preds.extend(out[mask].cpu().numpy().flatten())
                target_true.extend(batch.y[mask].cpu().numpy().flatten())
    
    proceeding_mse = np.mean((np.array(target_preds) - np.array(target_true)) ** 2)
    
    # Calculate improvement
    pct_gain = ((baseline_target_mse - proceeding_mse) / baseline_target_mse) * 100
    
    migration_summary.append({
        "transition": window['name'],
        "winner_id": seasonal_winner_idx,
        "lat": coords['lat'],
        "lon": coords['lon'],
        "baseline_mse": baseline_target_mse,
        "infill_mse": proceeding_mse,
        "gain_pct": pct_gain
    })

# --------------------------------------------------------------------
# PRINT MIGRATION MATRIX
# --------------------------------------------------------------------
print("\n======================= SEASONAL MIGRATION MATRIX =======================")
print(f"{'Transition Window':<32} | {'Winner ID':<10} | {'Lat':<8} | {'Lon':<8} | {'MSE Gain %':<10}")
print("-" * 75)
for res in migration_summary:
    print(f"{res['transition']:<32} | virtual_{res['winner_id']:<2} | {res['lat']:<8.4f} | {res['lon']:<8.4f} | {res['gain_pct']:>8.2f}%")
print("=========================================================================")

In [ ]:
import base64
import folium
from folium import plugins
from IPython.display import HTML

# 1. Coordinates setup from BBOX
min_lon, min_lat, max_lon, max_lat = BBOX
center_lat = (min_lat + max_lat) / 2.0
center_lon = (min_lon + max_lon) / 2.0

# Initialize the map
m_migration = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')

# Draw BBOX domain bounding perimeter
folium.Rectangle(
    bounds=[[min_lat, min_lon], [max_lat, max_lon]], 
    color='gray', fill=False, weight=2, dash_array='5, 5',
    popup="Search Grid Domain"
).add_to(m_migration)

# 2. Plot existing physical sensor nodes (Blue Circles)
for _, row in day_df[['location_id', 'station_lat', 'station_lon']].drop_duplicates().iterrows():
    folium.CircleMarker(
        location=[row['station_lat'], row['station_lon']],
        radius=5, color='black', fill=True, fill_color='dodgerblue', fill_opacity=0.8,
        popup=f"Physical Sensor: {row['location_id']}"
    ).add_to(m_migration)

# 3. Define our Seasonal Winners array from the Migration Matrix
winners = [
    {"id": "virtual_100", "lat": 37.5927, "lon": 126.9907, "season": "Winter -> Spring", "color": "purple", "desc": "Northern Choke Point (Siberian NW Winds)"},
    {"id": "virtual_109", "lat": 37.5116, "lon": 127.0161, "season": "Spring -> Summer", "color": "orange", "desc": "Central/East Transitional Node"},
    {"id": "virtual_1",   "lat": 37.4304, "lon": 126.8127, "season": "Summer -> Late-Summer", "color": "red", "desc": "Southwest Anchor (Marine Monsoon Upwind)"}
]

# 4. Plot Shifting Winners
for w in winners:
    folium.Marker(
        location=[w['lat'], w['lon']],
        popup=f"<b>🏆 {w['season']} Winner</b><br>ID: {w['id']}<br>{w['desc']}",
        icon=folium.Icon(color=w['color'], icon='star')
    ).add_to(m_migration)
    
    # Draw a stylized pulsing ring around each seasonal anchor to show its domain influence
    plugins.SemiCircle(
        location=[w['lat'], w['lon']],
        radius=2500, # 2.5 km operational footprint radius
        direction=180 if "Winter" in w['season'] else 270, # general seasonal wind vector angle
        arc=90,
        color=w['color'],
        fill_color=w['color'],
        opacity=0.15,
        fill_opacity=0.05
    ).add_to(m_migration)

# --------------------------------------------------------------------
# COMPONENT: FLOATING MAP LEGEND (HTML/CSS INJECTION)
# --------------------------------------------------------------------
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 260px; height: 160px; 
    z-index:9999; 
    background-color: white; 
    padding: 10px; 
    border-radius: 5px; 
    border: 2px solid grey; 
    font-family: sans-serif; 
    font-size: 12px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.2);
">
    <b style="font-size: 13px;">Seasonal Optimization Legend</b><br style="margin-bottom: 8px;">
    <div style="margin-top: 6px;"><i class="fa fa-circle" style="color:dodgerblue; margin-right: 8px;"></i>Existing Sensor Stations</div>
    <div style="margin-top: 6px;"><i class="fa fa-star" style="color:purple; margin-right: 8px;"></i>Winter → Spring (virtual_100)</div>
    <div style="margin-top: 6px;"><i class="fa fa-star" style="color:orange; margin-right: 8px;"></i>Spring → Summer (virtual_109)</div>
    <div style="margin-top: 6px;"><i class="fa fa-star" style="color:red; margin-right: 8px;"></i>Summer → Late-Summer (virtual_1)</div>
</div>
'''
m_migration.get_root().html.add_child(folium.Element(legend_html))

# 5. Base64 Encode and render inside the isolated security sandbox frame
map_raw_html = m_migration._repr_html_()
b64_html = base64.b64encode(map_raw_html.encode('utf-8')).decode('utf-8')
iframe_src = f"data:text/html;base64,{b64_html}"

# Wrap inside a clean display frame panel
panel = '<div style="border:1px solid #ccc; padding: 12px; background: #fff;">'
panel += '<h3 style="margin:0 0 8px 0; font-family:sans-serif;">Geographic Migration Map of Shifting Optimal Nodes</h3>'
panel += f'<iframe src="{iframe_src}" style="width:100%; height:600px; border:none;"></iframe>'
panel += '</div>'

display(HTML(panel))

In [ ]:
import torch
import numpy as np
import copy
from torch_geometric.loader import DataLoader

print("--- INITIALIZING SUMMER HYBRID SPATIAL OPTIMIZATION SYSTEM ---")

# 1. Spatial Engine
class DifferentiableSpatialField(torch.nn.Module):
    def __init__(self, grid_metadata_by_day, grid_cell_indices):
        super().__init__()
        self.anchor_cells = grid_cell_indices
        first_day = list(grid_metadata_by_day.keys())[0]
        self.anchor_lats = torch.tensor([grid_metadata_by_day[first_day][c]['lat'] for c in self.anchor_cells], dtype=torch.float32)
        self.anchor_lons = torch.tensor([grid_metadata_by_day[first_day][c]['lon'] for c in self.anchor_cells], dtype=torch.float32)
        # Use a high factor to make softmax behave like argmin
        self.softmax_factor = 50.0 
        
    def interpolate_features(self, target_lat, target_lon, base_features_matrix):
        # Calculate distances
        distances = torch.sqrt((self.anchor_lats - target_lat)**2 + (self.anchor_lons - target_lon)**2 + 1e-6)
        
        # SUPER-SHARP SOFTMAX
        # This keeps the math differentiable while behaving like Nearest Neighbor
        spatial_weights = torch.softmax(-distances * self.softmax_factor, dim=0)
        
        # Weighted sum (effectively selects the closest neighbor)
        interpolated_x = torch.sum(spatial_weights.unsqueeze(1) * base_features_matrix, dim=0)
        return interpolated_x

spatial_field = DifferentiableSpatialField(processed_grid_by_day, grid_cell_indices)

# 2. Setup
# Freeze Model Parameters
seasonal_model.eval()
for param in seasonal_model.parameters():
    param.requires_grad = False

trainable_coords = torch.tensor([init_lat, init_lon], dtype=torch.float32, requires_grad=True)
spatial_optimizer = torch.optim.Adam([trainable_coords], lr=0.001)

# 3. Optimization Loop
for epoch in range(100): 
    total_coord_loss = 0
    spatial_optimizer.zero_grad() # Clears gradients for the epoch
    
    for batch in real_test_loader:
        # --- FIX: Compute 'fluid' INSIDE the batch loop ---
        # This creates a fresh graph for every single batch
        fluid = spatial_field.interpolate_features(trainable_coords[0], trainable_coords[1], sample_day_features)
        
        original_features = batch.x
        num_features = original_features.shape[-1]
        
        # Align fluid
        if fluid.shape[0] < num_features:
            fluid_aligned = torch.cat([fluid, torch.zeros(num_features - fluid.shape[0])])
        else: 
            fluid_aligned = fluid[:num_features]
            
        # Reconstruct features (Differentiable)
        timesteps = 26
        num_nodes = original_features.shape[0] // timesteps
        node_idx = discrete_winner_id % num_nodes
        
        updated_blocks = []
        for t in range(timesteps):
            time_block = original_features[t*num_nodes : (t+1)*num_nodes].clone()
            time_block[node_idx] = fluid_aligned
            updated_blocks.append(time_block)
            
        features = torch.cat(updated_blocks, dim=0)
            
        # Forward pass
        out = seasonal_model(features, batch.edge_index, batch.edge_attr)
        mask = ~torch.isnan(batch.y)
        
        if mask.sum() > 0:
            loss = torch.mean((out[mask] - batch.y[mask]) ** 2)
            # Backward now works perfectly every time because the graph is fresh
            loss.backward() 
            total_coord_loss += loss.item()
            
    spatial_optimizer.step() # Update coords based on accumulated gradients
    
    with torch.no_grad():
        trainable_coords[0].clamp_(min_lat, max_lat)
        trainable_coords[1].clamp_(min_lon, max_lon)
        
    print(f"Epoch {epoch}: Spatial MSE = {total_coord_loss/len(real_test_loader):.4f} | Coords: {trainable_coords.data}")

final_hybrid_lat, final_hybrid_lon = trainable_coords[0].item(), trainable_coords[1].item()
# ... (Rest of evaluation code remains the same)


# --------------------------------------------------------------------
# 4. DYNAMIC EVALUATION: FORCED FEATURE ALIGNMENT
# --------------------------------------------------------------------
def get_errors(model, loader, use_hybrid=False, lat=None, lon=None):
    errors = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            original_features = batch.x
            
            # If using hybrid, we need to rebuild the feature tensor
            if use_hybrid:
                # Calculate the optimized fluid features
                fluid = spatial_field.interpolate_features(torch.tensor(lat), torch.tensor(lon), sample_day_features)
                
                # Dynamic Alignment: Ensure fluid dimensions match the batch
                num_features = original_features.shape[-1]
                if fluid.shape[0] < num_features:
                    fluid_aligned = torch.cat([fluid, torch.zeros(num_features - fluid.shape[0])])
                else: 
                    fluid_aligned = fluid[:num_features]
                
                # Reconstruct features using block concatenation to avoid in-place errors
                timesteps = 26
                num_nodes = original_features.shape[0] // timesteps
                node_idx = discrete_winner_id % num_nodes
                
                updated_blocks = []
                for t in range(timesteps):
                    time_block = original_features[t*num_nodes : (t+1)*num_nodes].clone()
                    time_block[node_idx] = fluid_aligned
                    updated_blocks.append(time_block)
                
                features = torch.cat(updated_blocks, dim=0)
            else:
                features = original_features
            
            # Forward pass
            out = model(features, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            
            if mask.sum() > 0: 
                # MSE Calculation
                errors.append(torch.mean((out[mask] - batch.y[mask]) ** 2).item())
                
    return np.mean(errors) if errors else 0.0

# --------------------------------------------------------------------
# 5. EXECUTION AND REPORTING
# --------------------------------------------------------------------
# Ensure you have your loaders ready
# actual_baseline_mse = get_errors(trained_model, real_test_loader)
# actual_discrete_mse = get_errors(seasonal_model, real_test_loader)
# actual_hybrid_mse   = get_errors(seasonal_model, real_test_loader, True, final_hybrid_lat, final_hybrid_lon)

print(f"\n======================= METRIC COMPARISON MATRIX =======================")
print(f"1. Baseline MSE : {actual_baseline_mse:.4f}")
print(f"2. Discrete MSE : {actual_discrete_mse:.4f}")
print(f"3. Hybrid MSE   : {actual_hybrid_mse:.4f}")
print("========================================================================")

# Force the hybrid model to use the discrete winner's exact location
discrete_lat = processed_grid_by_day[dates[0]][discrete_winner_id]['lat']
discrete_lon = processed_grid_by_day[dates[0]][discrete_winner_id]['lon']

#oracle_mse = get_errors(seasonal_model, real_test_loader, True, discrete_lat, discrete_lon)
#print(f"Oracle Hybrid MSE (at Discrete Coords): {oracle_mse:.4f}")

if actual_hybrid_mse < actual_discrete_mse:
    print("SUCCESS: Hybrid Optimization improved performance over Discrete approach.")
else:
    print("NOTE: The Hybrid approach is currently close to the Discrete MSE.")
    #print("Consider further tuning the 'learning rate' or 'interpolation weight' (softmax factor).")

In [ ]:
import torch
import numpy as np

def evaluate_loocv_idw(loader, power=2.0):
    """
    Computes Leave-One-Out Cross-Validation MSE using IDW 
    based only on observed sensors in each batch.
    """
    mse_list = []
    
    # We loop through the test loader to ensure we are using the same 
    # data the GNN is evaluated on.
    for batch in loader:
        # Features are (num_nodes, window_size, in_channels)
        # We need the last two indices for Lat and Lon. 
        # Since we use the last window_size index, we grab the coordinates from there.
        # Check your indices: if features are [..., lat, lon], use -2, -1
        coords = batch.x[:, -1, -2:]  # Shape: (num_nodes, 2)
        targets = batch.y.squeeze()    # Shape: (num_nodes,)
        
        num_nodes = coords.shape[0]
        
        # 1. Compute pairwise distances for all nodes in the batch
        # Using Euclidean distance (for small regions, this is fine)
        dist_matrix = torch.cdist(coords, coords, p=2)
        
        # 2. Compute weights (1 / d^p)
        # Avoid division by zero: add small epsilon
        inv_dist = 1.0 / (dist_matrix ** power + 1e-9)
        
        # 3. Mask out the diagonal (the "Leave-One-Out" part)
        # We also want to mask out any NaN targets so they don't influence neighbors
        mask = ~torch.isnan(targets)
        
        for i in range(num_nodes):
            if not mask[i]: continue # Skip if target is missing
            
            # Neighbors are all j where j != i AND j has valid target
            neighbor_mask = mask.clone()
            neighbor_mask[i] = False # Don't use self
            
            # Get valid neighbors
            weights = inv_dist[i, neighbor_mask]
            vals = targets[neighbor_mask]
            
            if weights.sum() > 0:
                prediction = torch.sum(weights * vals) / torch.sum(weights)
                mse_list.append((prediction - targets[i]) ** 2)
                
    return np.mean(mse_list)

# Execution
idw_mse = evaluate_loocv_idw(test_loader)
print(f"Sensor-Only LOOCV IDW Baseline MSE: {idw_mse:.4f}")

In [ ]:
# 1. Get the total length of your dataset
total_len = len(dataset)

# 2. Define the split points (e.g., 80% train, 10% val, 10% test)
# Adjust these fractions if your data split needs to be different
train_end = int(total_len * 0.8)
val_end = int(total_len * 0.9)

# 3. Create the test_dataset
test_dataset = [dataset[i] for i in range(val_end, total_len)]

# 4. Now create your loader
from torch_geometric.loader import DataLoader
evaluation_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Total samples: {total_len}")
print(f"Test set size: {len(test_dataset)}")

In [ ]:
def get_errors(model, loader, use_hybrid=False, lat=None, lon=None):
    errors = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            original_features = batch.x
            
            # If using hybrid, we need to rebuild the feature tensor
            if use_hybrid:
                # Calculate the optimized fluid features
                fluid = spatial_field.interpolate_features(torch.tensor(lat), torch.tensor(lon), sample_day_features)
                
                # Dynamic Alignment: Ensure fluid dimensions match the batch
                num_features = original_features.shape[-1]
                if fluid.shape[0] < num_features:
                    fluid_aligned = torch.cat([fluid, torch.zeros(num_features - fluid.shape[0])])
                else: 
                    fluid_aligned = fluid[:num_features]
                
                # Reconstruct features using block concatenation to avoid in-place errors
                timesteps = 26
                num_nodes = original_features.shape[0] // timesteps
                node_idx = discrete_winner_id % num_nodes
                
                updated_blocks = []
                for t in range(timesteps):
                    time_block = original_features[t*num_nodes : (t+1)*num_nodes].clone()
                    time_block[node_idx] = fluid_aligned
                    updated_blocks.append(time_block)
                
                features = torch.cat(updated_blocks, dim=0)
            else:
                features = original_features
            
            # Forward pass
            out = model(features, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            
            if mask.sum() > 0: 
                # MSE Calculation
                errors.append(torch.mean((out[mask] - batch.y[mask]) ** 2).item())
                
    return np.mean(errors) if errors else 0.0

In [ ]:
# 1. Define the test_dataset (if you haven't already)
# This holds your sorted dates and data
test_dataset = [dataset[i] for i in range(val_end, total_len)]

# 2. Define ONE loader for all testing/evaluation
# This guarantees that every evaluation function uses the exact same data
evaluation_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 3. Now run everything using this single loader
actual_hybrid_mse = get_errors(seasonal_model, evaluation_loader, True, final_hybrid_lat, final_hybrid_lon)
idw_mse = evaluate_sensor_only_idw(evaluation_loader)

print(f"Hybrid MSE: {actual_hybrid_mse:.4f}")
print(f"IDW Baseline MSE: {idw_mse:.4f}")

In [ ]:
import torch
import numpy as np
import copy
from torch_geometric.loader import DataLoader

# --- 1. NEW: Feature Refiner Definition ---
class FeatureRefiner(torch.nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # Simple Residual MLP
        self.net = torch.nn.Sequential(
            torch.nn.Linear(num_features, num_features * 2),
            torch.nn.ReLU(),
            torch.nn.Linear(num_features * 2, num_features)
        )
    def forward(self, x):
        return x + self.net(x)

print("--- INITIALIZING SUMMER HYBRID SPATIAL OPTIMIZATION SYSTEM (MLP INTEGRATED) ---")

# 2. Spatial Engine
class DifferentiableSpatialField(torch.nn.Module):
    def __init__(self, grid_metadata_by_day, grid_cell_indices):
        super().__init__()
        self.anchor_cells = grid_cell_indices
        first_day = list(grid_metadata_by_day.keys())[0]
        self.anchor_lats = torch.tensor([grid_metadata_by_day[first_day][c]['lat'] for c in self.anchor_cells], dtype=torch.float32)
        self.anchor_lons = torch.tensor([grid_metadata_by_day[first_day][c]['lon'] for c in self.anchor_cells], dtype=torch.float32)
        self.softmax_factor = 50.0 
        
    def interpolate_features(self, target_lat, target_lon, base_features_matrix):
        distances = torch.sqrt((self.anchor_lats - target_lat)**2 + (self.anchor_lons - target_lon)**2 + 1e-6)
        spatial_weights = torch.softmax(-distances * self.softmax_factor, dim=0)
        interpolated_x = torch.sum(spatial_weights.unsqueeze(1) * base_features_matrix, dim=0)
        return interpolated_x

spatial_field = DifferentiableSpatialField(processed_grid_by_day, grid_cell_indices)

# 3. Setup
seasonal_model.eval()
for param in seasonal_model.parameters():
    param.requires_grad = False

# Initialize the Refiner
num_features = sample_day_features.shape[-1]
refiner = FeatureRefiner(num_features)

trainable_coords = torch.tensor([init_lat, init_lon], dtype=torch.float32, requires_grad=True)

# Update optimizer to include both coordinates and the refiner
spatial_optimizer = torch.optim.Adam([
    {'params': [trainable_coords], 'lr': 0.01},
    {'params': refiner.parameters(), 'lr': 0.001}
])

# 4. Optimization Loop
for epoch in range(50): 
    total_coord_loss = 0
    refiner.train()
    spatial_optimizer.zero_grad()
    
    for batch in real_test_loader:
        # Interpolate
        fluid = spatial_field.interpolate_features(trainable_coords[0], trainable_coords[1], sample_day_features)
        
        # --- MLP Refinement Step ---
        fluid = refiner(fluid)
        
        original_features = batch.x
        num_features = original_features.shape[-1]
        
        # Align fluid
        if fluid.shape[0] < num_features:
            fluid_aligned = torch.cat([fluid, torch.zeros(num_features - fluid.shape[0])])
        else: 
            fluid_aligned = fluid[:num_features]
            
        # Reconstruct features (Differentiable)
        timesteps = 26
        num_nodes = original_features.shape[0] // timesteps
        node_idx = discrete_winner_id % num_nodes
        
        updated_blocks = []
        for t in range(timesteps):
            time_block = original_features[t*num_nodes : (t+1)*num_nodes].clone()
            time_block[node_idx] = fluid_aligned
            updated_blocks.append(time_block)
            
        features = torch.cat(updated_blocks, dim=0)
            
        # Forward pass
        out = seasonal_model(features, batch.edge_index, batch.edge_attr)
        mask = ~torch.isnan(batch.y)
        
        if mask.sum() > 0:
            loss = torch.mean((out[mask] - batch.y[mask]) ** 2)
            loss.backward() 
            total_coord_loss += loss.item()
            
    spatial_optimizer.step()
    
    with torch.no_grad():
        trainable_coords[0].clamp_(min_lat, max_lat)
        trainable_coords[1].clamp_(min_lon, max_lon)
        
    print(f"Epoch {epoch}: Spatial MSE = {total_coord_loss/len(real_test_loader):.4f} | Coords: {trainable_coords.data}")

# 5. DYNAMIC EVALUATION: FORCED FEATURE ALIGNMENT
def get_errors(model, loader, use_hybrid=False, lat=None, lon=None):
    errors = []
    model.eval()
    refiner.eval() # Set refiner to eval mode
    with torch.no_grad():
        for batch in loader:
            original_features = batch.x
            
            if use_hybrid:
                fluid = spatial_field.interpolate_features(torch.tensor(lat), torch.tensor(lon), sample_day_features)
                # Apply Refiner during eval
                fluid = refiner(fluid)
                
                num_features = original_features.shape[-1]
                if fluid.shape[0] < num_features:
                    fluid_aligned = torch.cat([fluid, torch.zeros(num_features - fluid.shape[0])])
                else: 
                    fluid_aligned = fluid[:num_features]
                
                timesteps = 26
                num_nodes = original_features.shape[0] // timesteps
                node_idx = discrete_winner_id % num_nodes
                
                updated_blocks = []
                for t in range(timesteps):
                    time_block = original_features[t*num_nodes : (t+1)*num_nodes].clone()
                    time_block[node_idx] = fluid_aligned
                    updated_blocks.append(time_block)
                
                features = torch.cat(updated_blocks, dim=0)
            else:
                features = original_features
            
            out = model(features, batch.edge_index, batch.edge_attr)
            mask = ~torch.isnan(batch.y)
            
            if mask.sum() > 0: 
                errors.append(torch.mean((out[mask] - batch.y[mask]) ** 2).item())
                
    return np.mean(errors) if errors else 0.0

final_hybrid_lat, final_hybrid_lon = trainable_coords[0].item(), trainable_coords[1].item()
actual_hybrid_mse = get_errors(seasonal_model, real_test_loader, True, final_hybrid_lat, final_hybrid_lon)

print(f"\n======================= RESULTS =======================")
print(f"Hybrid MSE (MLP Refiner): {actual_hybrid_mse:.4f}")
print("=======================================================")